# Análisis exploratorio de incidentes viales en CDMX (2024)

Anteriormente, en nuestro [notebook de limpieza](01_Data_Understanding_And_Cleaning.ipynb) revisamos los cinco archivos de la SSC que contienen información acerca de los hechos de tránsito, corregimos inconsistencias, limpiamos y conservamos completos los registros de **30,655 incidentes** que contenía la fuente original. Ahora exploraremos los datos limpios para poder descubrir qué nos pueden contar estos registros.

Comenzaremos viendo como se distribuyen los accidentes a lo largo de los meses, seguiremos con información de los vehículos involucrados y de las personas afectadas, también exploraremos las características de las vialidades y la distribución geográfica.

La idea es avanzar a partir de preguntas intuitivas e ir describiendo lo que nos permitan observar los datos, algunas intuiciones se sostendrán a partir de los resultados y seguramente otras cambiarán a lo largo de la exploración.

**Fuente:** [Hechos de tránsito registrados por la SSC, 2024 — serie ampliada](https://datos.cdmx.gob.mx/dataset/hechos-de-transito-registrados-por-la-ssc-2024-serie-de-datos-ampliada-no-comparativa).

In [ ]:
# Importamos librerias
import pandas as pd
from pathlib import Path
from google.colab import drive

# Cargamos drive para leer archivos y definimos rutas
drive.mount('/content/drive')
ruta_proyecto = Path('/content/drive/MyDrive/ProyectosPython/cdmx-road-safety-intelligence')
ruta_processed = ruta_proyecto / 'data/processed'

# Cargamos archivos en DFs dentro de un diccionario
dfs = {archivo.stem.removesuffix('_clean'): pd.read_parquet(archivo)
       for archivo in sorted(ruta_processed.glob('*_clean.parquet'))}

for nombre, df in dfs.items():
    print(nombre, df.shape)

## Datos que utilizaremos

Cargamos los cinco archivos Parquet preparados en la limpieza. Cada uno contiene un registro por `folio`, pero aporta información distinta: El archivo de `hechos_ssc` describe los datos generales del incidente y sus consecuencias, el de `vehiculos`, información sobre el vehículo involucrado en el hecho y los tres archivos restantes, atributos de las personas afectadas.

Ya que los folios coinciden entre archivos, los utilizaremos para relacionarlos.

## ¿Cómo se distribuyen los incidentes durante 2024?

Partiremos a partir de lo más general que es el número de incidentes registrados en cada mes.

In [ ]:
# Trabajaremos con hechos_ssc
df = dfs['hechos_ssc']

# Extraemos mes de fecha_evento y contamos los incidentes por mes.
hechos_por_mes = df['fecha_evento'].dt.month.value_counts().sort_index()

# Diccionario de mes
meses = {
    1: 'Enero',
    2: 'Febrero',
    3: 'Marzo',
    4: 'Abril',
    5: 'Mayo',
    6: 'Junio',
    7: 'Julio',
    8: 'Agosto',
    9: 'Septiembre',
    10: 'Octubre',
    11: 'Noviembre',
    12: 'Diciembre',
}

hechos_por_mes.rename(index=meses, inplace=True)

# Construir gráfica de barras

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 6))
ax = sns.barplot(x=hechos_por_mes.index, y=hechos_por_mes.values)
ax.bar_label(ax.containers[0], padding=3)
plt.title('Incidentes viales por mes - CDMX 2024')
plt.xlabel('Mes')
plt.xticks(rotation=45)
plt.ylabel('Número de incidentes')
plt.show()

Los totales mensuales no cambian demasiado, ahora veremos si este comportamiento se mantiene similar a lo largo de las 16 alcaldías.

In [ ]:
# Contamos los incidentes por alcaldía y mes

incidentes_alcaldia_mes = pd.crosstab(df['alcaldia_utilizada'], df['fecha_evento'].dt.month)

incidentes_alcaldia_mes.rename(columns=meses, inplace=True)

display(incidentes_alcaldia_mes)

### Incidentes por alcaldía y mes

Primero comparamos los conteos absolutos, para identificar dónde se concentra el mayor volumen de registros.

In [ ]:
plt.figure(figsize=(8, 6))

sns.heatmap( incidentes_alcaldia_mes, annot=True, fmt='d', cmap='YlOrRd', linewidths=0.5)

plt.title('Incidentes viales por alcaldía y mes - CDMX 2024')
plt.xlabel('Mes')
plt.ylabel('Alcaldía')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Ahora revisaremos conteos relativos por alcaldia (las filas sumarán 100%) para poder saber de todos los incidentes que registrados dentro de **cada alcaldía** durante el año, ¿qué porcentaje ocurrió en cada mes?.

Esto ayudará a disminuir el impacto por volumen total de accidentes, estableciendo límites mínimos y máximos internos por alcaldia permitiendo comparar como varió el total de accidentes dentro de cada una.

In [ ]:
from matplotlib.colors import ListedColormap

# Porcentaje mensual respecto al total anual de cada alcaldía
incidentes_alcaldia_mes_pct = (incidentes_alcaldia_mes.div(incidentes_alcaldia_mes.sum(axis=1), axis=0).mul(100))

# Clasificamos los meses según su posición dentro de cada alcaldía
heatmap_rangos = incidentes_alcaldia_mes_pct.rank(axis=1, method='average', pct=True)

# Paleta con siete niveles de intensidad
colores = ListedColormap(sns.color_palette('YlOrRd', 7))

plt.figure(figsize=(8, 6))

sns.heatmap(
    heatmap_rangos,
    annot=incidentes_alcaldia_mes_pct,
    fmt='.1f',
    cmap=colores,
    vmin=0,
    vmax=1,
    linewidths=0.5,
    cbar_kws={'label': 'Posición relativa del mes en cada alcaldía'}
)

plt.title('Distribución mensual de incidentes por alcaldía - CDMX 2024')
plt.xlabel('Mes')
plt.ylabel('Alcaldía')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Diferencias mensuales entre alcaldías

En el último heatmap, **las cifras son porcentajes** y el color indica la **posición relativa del mes dentro de su alcaldía**. Así evitamos que las zonas con mayor volumen dominen toda la escala, aunque las diferencias pequeñas pueden hacer que el cambio se muestre más intenso de lo que es.

Al comparar la distribución dentro de cada alcaldía, aparecen matices que no se ven cuando se visualiza el conteo del total de incidentes en la ciudad. Por ejemplo, Cuajimalpa tiene meses relativamente altos entre agosto y octubre, mientras que Tláhuac destaca más al inicio del año.

A continuación veremos cuáles fueron los tipos de vehículos involucrados en los hechos.

In [ ]:
df_vehiculos = dfs['vehiculos'].copy()

incidentes_vehiculo_mes = pd.crosstab(df_vehiculos['tipo_vehiculo'], df_vehiculos['fecha_evento'].dt.month)

incidentes_vehiculo_mes.rename(columns=meses, inplace=True)

display(incidentes_vehiculo_mes)

## ¿Qué vehículos aparecen en los incidentes?

Las motocicletas llaman la atención desde el primer conteo ya que están involucradaas en un gran número de acidentes. Revisaremos su porcentaje de participación y la del resto de las categorías de vehiculos.

In [ ]:
resumen_vehiculos = pd.DataFrame({'Incidentes': incidentes_vehiculo_mes.sum(axis=1)})

resumen_vehiculos['Porcentaje'] = (resumen_vehiculos['Incidentes'] / resumen_vehiculos['Incidentes'].sum() * 100)

resumen_vehiculos = resumen_vehiculos.sort_values('Incidentes', ascending=False)

display(resumen_vehiculos)

In [ ]:
resumen_vehiculos_plot = resumen_vehiculos.sort_values('Incidentes', ascending=False)

plt.figure(figsize=(9, 5))
sns.barplot(data=resumen_vehiculos_plot, x='Incidentes', y=resumen_vehiculos_plot.index)
plt.title('Incidentes registrados por tipo de vehículo')
plt.xlabel('Número de incidentes')
plt.ylabel('Tipo de vehículo')
plt.tight_layout()
plt.show()

In [ ]:
# Porcentaje de incidentes por tipo de vehículo
plt.figure(figsize=(9, 5))
sns.barplot(data=resumen_vehiculos_plot, x='Porcentaje', y=resumen_vehiculos_plot.index)
plt.title('Porcentaje de incidentes registrados por tipo de vehículo')
plt.xlabel('Porcentaje del total de incidentes')
plt.ylabel('Tipo de vehículo')
plt.tight_layout()
plt.show()

Las motocicletas aparecen en **17,391 incidentes (56.73%)**, seguidas por los automóviles, con 6,239 (20.35%). Ya que `vehiculos` registra un único tipo de vehículo por folio, no tenemos un inventario de todos los participantes del accidente, sólo el registrado como `Vehículo 1`.

### Agrupación de vehículos

Para comparar categorías, crearemos grupos al interés de esta exploración, agrupando los vehículos en: motocicletas, micromovilidad, transporte de personas, transporte colectivo y carga. Conservamos `Sin dato` por separado.

Esta es una **clasificación que construimos para este análisis**, no una categoría oficial de la fuente. Por ejemplo, incluimos automóviles, camionetas y taxis en transporte de personas, y en micromovilidad las bicicletas y monopatínes.

In [ ]:
df_vehiculos = dfs['vehiculos'].copy()

# Definimos los grupos
mapa_grupos_vehiculo = {
    'AUTOMOVIL': 'Transporte de personas',
    'CAMIONETA': 'Transporte de personas',
    'TAXI': 'Transporte de personas',
    'MOTOCICLETA': 'Motocicleta',
    'BICICLETA': 'Micromovilidad',
    'MONOPATIN': 'Micromovilidad',
    'AUTOBUS DE PASAJEROS': 'Transporte colectivo',
    'MICROBUS': 'Transporte colectivo',
    'METROBUS': 'Transporte colectivo',
    'TROLEBUS': 'Transporte colectivo',
    'TREN LIGERO': 'Transporte colectivo',
    'CAMION DE CARGA': 'Carga',
    'SD': 'Sin dato'
}

df_vehiculos['tipo_vehiculo_grupo'] = df_vehiculos['tipo_vehiculo'].map(mapa_grupos_vehiculo)

In [ ]:
# Resúmenes por grupo
resumen_grupos = (df_vehiculos['tipo_vehiculo_grupo'].value_counts().rename_axis('Grupo').reset_index(name='Incidentes'))

resumen_grupos['Porcentaje'] = (resumen_grupos['Incidentes'] / resumen_grupos['Incidentes'].sum() * 100)

display(resumen_grupos)

In [ ]:
# Incidentes por grupo
plt.figure(figsize=(9, 5))
sns.barplot(data=resumen_grupos, x='Incidentes', y='Grupo')

plt.title('Incidentes registrados por grupo de vehículo')
plt.xlabel('Número de incidentes')
plt.ylabel('Grupo de vehículo')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(data=resumen_grupos, x='Porcentaje', y='Grupo')

plt.title('Porcentaje de incidentes por grupo de vehículo')
plt.xlabel('Porcentaje del total')
plt.ylabel('Grupo de vehículo')
plt.tight_layout()
plt.show()

Después de agrupar categorías podemos notar que las motocicletas siguen siendo el grupo mayoritario de vehiculos que participan en los accidentes. Veamos si todos los grupos presentan la misma distribución a lo largo del año.

In [ ]:
# Incidentes por grupo de vehículo y mes
incidentes_grupo_mes = pd.crosstab(df_vehiculos['tipo_vehiculo_grupo'], df_vehiculos['fecha_evento'].dt.month)

incidentes_grupo_mes.rename(columns=meses, inplace=True)

incidentes_grupo_mes = incidentes_grupo_mes.loc[incidentes_grupo_mes.sum(axis=1).sort_values(ascending=False).index]

display(incidentes_grupo_mes)

Los conteos mensuales sugieren algunas diferencias, pero los grupos tienen tamaños muy distintos y los meses no duran lo mismo. Por lo que compararemos el promedio diario de cada mes con el promedio diario anual del propio grupo (índice 100).

In [ ]:
import calendar

# Cantidad de días de cada mes de 2024
dias_por_mes = pd.Series({meses[m]: calendar.monthrange(2024, m)[1] for m in range(1, 13)})

# Incidentes promedio por día en cada mes
promedio_diario = incidentes_grupo_mes.div(dias_por_mes, axis=1)

promedio_anual = incidentes_grupo_mes.sum(axis=1) / 366

# Índice 100 = promedio diario anual de cada grupo
indice_mensual = promedio_diario.div(promedio_anual, axis=0) * 100

print("===== INDICE MENSUAL POR GRUPO ===== \n")
display(indice_mensual.round(1))

In [ ]:
# Excluimos los registros "Sin Dato"
indice_plot = indice_mensual.drop(index='Sin dato')

plt.figure(figsize=(9, 5))

for grupo, valores in indice_plot.iterrows():
    plt.plot(valores.index, valores.values, marker='o', linewidth=2, label=grupo)

# Referencia = promedio diario anual de cada grupo
plt.axhline(
    y=100,
    color='gray',
    linestyle='--',
    linewidth=1,
    label='Promedio anual'
)

plt.title('Variación mensual de incidentes por grupo de vehículo - CDMX 2024')
plt.xlabel('Mes')
plt.ylabel('Índice respecto al promedio diario anual (100 = promedio)')
plt.xticks(rotation=45)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

Separamos los grupos en graficos distintos para ver más claras las variaciones

In [ ]:
fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(8, 6), sharex=True, sharey=True)

axes = axes.flatten()

# Lista de colores
colores = ['#2E5B88', '#D96B43', '#2E8B57', '#8B2E5F', '#D9A743']

for i, (ax, (grupo, valores)) in enumerate(zip(axes, indice_plot.iterrows())):

    # Asignamos el color
    ax.plot(valores.index, valores.values, marker='o', linewidth=2, color=colores[i])

    # Promedio anual
    ax.axhline(100, color=colores[i], linestyle='--', linewidth=1, alpha=0.6)

    ax.set_title(grupo)
    ax.set_ylim(60, 130)
    ax.grid(axis='y', alpha=0.25)
    ax.tick_params(axis='x', rotation=45)

# Ocultamos el sexto panel que no utilizamos
axes[-1].axis('off')

fig.suptitle('Variación mensual de incidentes por grupo de vehículo - CDMX 2024', fontsize=14)
fig.supxlabel('Mes')
fig.supylabel('Índice respecto al promedio diario anual (100 = promedio)')
plt.tight_layout()
plt.show()

### Variación mensual por grupo de vehículo

Al comparar cada grupo contra su propio promedio diario anual, aparecen patrones distintos. Las motocicletas muestran una disminución durante julio y agosto, la micromovilidad presenta valores relativamente altos entre marzo y junio, el transporte de personas destaca en febrero y el colectivo en abril.

El índice 100 evita que se introduzca confusión en la variación temporal por el tamaño de un grupo.

## ¿Los vehículos con más incidentes también concentran más víctimas?

Hasta ahora solo hemos contamos incidentes, pero un sólo evento puede dejar una o varias personas afectadas. Asi que uniremos el tipo de vehículo registrado con los datos de `personas_lesionadas` y `personas_fallecidas`, conservando un folio por incidente.

In [ ]:
# Seleccionamos únicamente las columnas necesarias
df_consecuencias = dfs['hechos_ssc'][['folio', 'personas_lesionadas', 'personas_fallecidas']].merge(
    df_vehiculos[['folio', 'tipo_vehiculo', 'tipo_vehiculo_grupo']], on='folio', how='inner', validate='one_to_one')

display(df_consecuencias.head())

In [ ]:
# Verificamos integridad de la unión

print('Registros originales:', len(dfs['hechos_ssc']))
print('Registros unidos:', len(df_consecuencias))
print('Folios únicos:', df_consecuencias['folio'].nunique())
print('Folios duplicados:', df_consecuencias['folio'].duplicated().sum())
print('\nTotales de víctimas:')
print(df_consecuencias[['personas_lesionadas', 'personas_fallecidas']].sum())

La unión conservó los **30,655 folios** sin duplicados. Ahora podemos comparar las consecuencias asociadas con cada grupo de vehículo.

In [ ]:
resumen_consecuencias = (
    df_consecuencias.groupby('tipo_vehiculo_grupo').agg(
        incidentes=('folio', 'nunique'),
        lesionados=('personas_lesionadas', 'sum'),
        fallecidos=('personas_fallecidas', 'sum')
    )
    .sort_values('incidentes', ascending=False)
)

display(resumen_consecuencias)

Para comparar grupos de tamaños distintos, calcularemos cuántas personas resultaron lesionadas o fallecieron **por cada 100 incidentes registrados**.

In [ ]:
resumen_consecuencias['lesionados_por_100'] = resumen_consecuencias['lesionados'] / resumen_consecuencias['incidentes'] * 100

resumen_consecuencias['fallecidos_por_100'] = resumen_consecuencias['fallecidos'] / resumen_consecuencias['incidentes'] * 100

display(resumen_consecuencias.round(2))

El transporte colectivo presenta más personas lesionadas por incidente, mientras que el de carga destaca por el índice de fallecidos por cada 100 incidentes. La ocupación o las características de los vehículos podrían ayudar a entenderlo, pero todavía no sabemos quiénes eran las víctimas ni qué otros vehículos participaron.

In [ ]:
# Personas lesionadas por cada 100 incidentes
datos_plot = resumen_consecuencias.drop(index='Sin dato').sort_values('lesionados_por_100')

plt.figure(figsize=(8, 4))

sns.barplot(data=datos_plot, x='lesionados_por_100', y=datos_plot.index, color='steelblue')
plt.title('Personas lesionadas por cada 100 incidentes - CDMX 2024')
plt.xlabel('Personas lesionadas por cada 100 incidentes')
plt.ylabel('Grupo de vehículo')
plt.tight_layout()
plt.show()

Revisemos ahora las personas fallecidas por cada 100 incidentes.

In [ ]:
datos_plot = (resumen_consecuencias.drop(index='Sin dato').sort_values('fallecidos_por_100'))

plt.figure(figsize=(8, 4))

sns.barplot(data=datos_plot, x='fallecidos_por_100', y=datos_plot.index, color='indianred')

plt.title('Personas fallecidas por cada 100 incidentes - CDMX 2024')
plt.xlabel('Personas fallecidas por cada 100 incidentes')
plt.ylabel('Grupo de vehículo')
plt.tight_layout()
plt.show()

Al ver la diferencia en lesionados dentro de los grupos aparece la siguiente pregunta: ¿en qué grupos son más frecuentes los incidentes con **dos o más personas lesionadas**?

In [ ]:
# Identificamos incidentes con múltiples personas lesionadas
df_consecuencias['multiples_lesionados'] = df_consecuencias['personas_lesionadas'] >= 2

resumen_multiples = (
    df_consecuencias.groupby('tipo_vehiculo_grupo').agg(
        incidentes=('folio', 'count'),
        incidentes_multiples=('multiples_lesionados', 'sum')
        )
    )

resumen_multiples['porcentaje_multiples'] = resumen_multiples['incidentes_multiples'] / resumen_multiples['incidentes'] * 100

display(resumen_multiples.sort_values('porcentaje_multiples', ascending=False).round(2)
)

El transporte colectivo encabeza la proporción de incidentes con dos o más lesionados: **22.47%**, frente al **11.79%** de las motocicletas. Esta diferencia podría ser compatible con la hipótesis de que este tipo de transporte tiene una mayor cantidad de personas expuestas por evento, pero no podemos verificarlo ya que los datos no identifican a los ocupantes de cada vehículo involucrado.

In [ ]:
# Frecuencia de incidentes según el número de personas lesionadas
distribucion_lesionados = pd.crosstab(df_consecuencias['tipo_vehiculo_grupo'], df_consecuencias['personas_lesionadas'])
display(distribucion_lesionados)

La distribución cambia en los extremos, los incidentes asociados con motocicletas suelen registrar una persona lesionada, mientras que el transporte colectivo presenta una cola más larga de eventos con varios lesionados.

Encontramos además un registro de carga con **42 personas lesionadas**. Antes de sacar conclusiones sobre promedios, revisaremos ese caso.

In [ ]:
# Buscamos outliers con más de 20 lesionados
df_consecuencias.loc[df_consecuencias['personas_lesionadas'] >= 20]

In [ ]:
# Investigamos el folio en nuestros datos
folio_extremo = 'C5/20240515/01841'

print(dfs['hechos_ssc'].loc[lambda df: df['folio'] == folio_extremo].T)

Se realizó un a búsqueda en internet y redes sociales con los datos del registro para encontrar alguna noticia que pudiera dar un poco de contexto, sin embargo, no encontramos información externa suficiente para confirmar el evento de 42 lesionados. Por lo que conservaremos el registro tal como aparece en la fuente y calcularemos cuánto cambia el indicador si lo excluimos.

In [ ]:
# Promedio de lesionados por cada 100 incidentes de carga
carga = df_consecuencias[df_consecuencias['tipo_vehiculo_grupo'] == 'Carga']

tasa_original = carga['personas_lesionadas'].sum() / len(carga) * 100

# Análisis de sensibilidad sin el registro extremo
carga_sin_extremo = carga[carga['folio'] != 'C5/20240515/01841']

tasa_sin_extremo = carga_sin_extremo['personas_lesionadas'].sum() / len(carga_sin_extremo) * 100

print(f'Con todos los registros: {tasa_original:.2f}')
print(f'Sin el caso extremo:    {tasa_sin_extremo:.2f}')

### Registro extremo en vehículos de carga

El folio `C5/20240515/01841` registra 42 lesionados. Al excluirlo, el indicador de carga pasa de **123.24 a 115.26 lesionados por cada 100 incidentes**. Mostrando que el caso influye en el promedio, pero no cambia el hallazgo principal ya que el transporte colectivo sigue siendo el que tiene más lesionados por incidente.

Una forma de estudiar la cola de la distribución, sin depender tanto de un único extremo es contar los incidentes con **cinco o más personas lesionadas**.

In [ ]:
# Indicador de incidentes con cinco o más lesionados
df_consecuencias['cinco_o_mas_lesionados'] = df_consecuencias['personas_lesionadas'] >= 5

resumen_cinco = df_consecuencias.groupby('tipo_vehiculo_grupo')['cinco_o_mas_lesionados'].agg(['sum', 'count', 'mean'])
resumen_cinco['porcentaje'] = resumen_cinco['mean'] * 100

display(resumen_cinco[['sum', 'count', 'porcentaje']].sort_values('porcentaje', ascending=False).round(2))

In [ ]:
# Incidentes con cinco o más lesionados
datos_plot = resumen_cinco.drop(index='Sin dato').sort_values('porcentaje')

fig, ax = plt.subplots(figsize=(8, 4))

# Líneas de referencia desde cero hasta cada porcentaje
ax.hlines(y=datos_plot.index, xmin=0, xmax=datos_plot['porcentaje'], color='lightgray', linewidth=2)

# Puntos que representan cada porcentaje
ax.scatter(x=datos_plot['porcentaje'], y=datos_plot.index, color='#C56C54', s=100, zorder=3)

# Etiquetas numéricas
for grupo, porcentaje in datos_plot['porcentaje'].items():
    ax.annotate(
        f'{porcentaje:.2f}%',
        xy=(porcentaje, grupo),
        xytext=(8, 0),
        textcoords='offset points',
        va='center'
    )

# Limite del eje x apropiado
ax.set_xlim(0, datos_plot['porcentaje'].max() * 1.25)
ax.set_title('Incidentes con 5 o más lesionados - CDMX 2024')
ax.set_xlabel('Porcentaje de incidentes del grupo')
ax.set_ylabel('Grupo de vehículo')
ax.grid(axis='x', alpha=0.2)

plt.tight_layout()
plt.show()

### Incidentes con múltiples lesionados

La diferencia se acentúa al revisar que porcentaje del total representan los eventos con cinco o más lesionados: **5.82%** en transporte colectivo, **0.96%** en transporte de personas y **0.06%** en motocicletas.

Aqui podría intervenir la ocupación de los vehículos, pero como no contamos con información sobre los ocupantes de cada vehículo, aqui sólo contabilizamos a **todas las personas lesionadas en el incidente**.

## ¿Quiénes aparecen como personas afectadas?

Queremos explorar el tipo de participante en cada hecho, asi como su sexo y edad. Los datos están en tres archivos relacionados por su `folio`, pero cada registro conserva **una sola fila por incidente**, incluso cuando hubo varias víctimas. Antes de tratar de interpretar perfiles individuales, tenemos que comprobar cómo están representados los datos.

In [ ]:
columnas = {
    'lesionados_tipo_persona': [
        'folio', 'tipo_persona', 'condicion_persona', 'total'
    ],
    'lesionados_tipo_edad': [
        'folio', 'edad', 'sexo', 'condicion_persona'
    ],
    'lesionados_tipo_sexo': [
        'folio', 'sexo', 'condicion_persona', 'total'
    ]
}

for nombre, cols in columnas.items():
    print(f'\n{nombre}')
    print(dfs[nombre][cols].head(10))

In [ ]:
edad = dfs['lesionados_tipo_edad'].set_index('folio')
sexo = dfs['lesionados_tipo_sexo'].set_index('folio')

comparacion = pd.DataFrame({
    'sexo_edad': edad['sexo'],
    'sexo_total': sexo['sexo'],
    'condicion_edad': edad['condicion_persona'],
    'condicion_total': sexo['condicion_persona'],
    'total_personas': sexo['total']
})

print('Diferencias en sexo:',
      (comparacion['sexo_edad'] != comparacion['sexo_total']).sum())

print('Diferencias en condición:',
      (comparacion['condicion_edad'] != comparacion['condicion_total']).sum())

print('\nDistribución de total_personas:')
print(comparacion['total_personas'].value_counts().sort_index())

In [ ]:
persona = dfs['lesionados_tipo_persona'].set_index('folio')
sexo = dfs['lesionados_tipo_sexo'].set_index('folio')
edad = dfs['lesionados_tipo_edad'].set_index('folio')

comparacion_personas = pd.DataFrame({
    'tipo_persona': persona['tipo_persona'],
    'total_tipo_persona': persona['total'],
    'total_tipo_sexo': sexo['total'],
    'sexo': sexo['sexo'],
    'edad': edad['edad'],
    'condicion_persona': persona['condicion_persona']
})

# Comprobamos si los conteos coinciden
diferencias = (comparacion_personas['total_tipo_persona'] != comparacion_personas['total_tipo_sexo'])

print('Folios con conteos diferentes:', diferencias.sum())

print(comparacion_personas.loc[diferencias].head(15))

In [ ]:
comparacion_totales = comparacion_personas.join(dfs['hechos_ssc'].set_index('folio')[['personas_lesionadas', 'personas_fallecidas']], validate='one_to_one')

# ¿Total representa lesionados, fallecidos o ambos?
comparacion_totales['total_victimas'] = (comparacion_totales['personas_lesionadas'] + comparacion_totales['personas_fallecidas'])

print('Coincide con lesionados:', (comparacion_totales['total_tipo_persona'] == comparacion_totales['personas_lesionadas']).sum())

print('Coincide con fallecidos:', (comparacion_totales['total_tipo_persona'] == comparacion_totales['personas_fallecidas']).sum())

print('Coincide con lesionados + fallecidos:', (comparacion_totales['total_tipo_persona'] == comparacion_totales['total_victimas']).sum())

In [ ]:
# Combinación de consecuencias del incidente
condiciones = comparacion_totales.assign(
    consecuencias=lambda df: (
        df['personas_lesionadas'].gt(0).map({True: 'Con lesionados', False: 'Sin lesionados'})
        + ' / ' +
        df['personas_fallecidas'].gt(0).map({True: 'Con fallecidos', False: 'Sin fallecidos'})
    )
)

pd.crosstab(
    condiciones['condicion_persona'],
    condiciones['consecuencias']
)

In [ ]:
# Incidentes con exactamente una persona lesionada y ningún fallecimiento
casos_individuales = comparacion_totales.loc[(comparacion_totales['personas_lesionadas'] == 1) & (comparacion_totales['personas_fallecidas'] == 0)].copy()

print('Incidentes seleccionados:', len(casos_individuales))

print(casos_individuales[['tipo_persona', 'sexo', 'edad', 'condicion_persona']].head(10))

In [ ]:
# Revisamos valores faltantes y categorías desconocidas
print('Valores faltantes:')
print(casos_individuales[['tipo_persona', 'sexo', 'edad']].isna().sum())

print('\nDistribución del sexo:')
print(casos_individuales['sexo'].value_counts(dropna=False))

print('\nDistribución del tipo de persona:')
print(casos_individuales['tipo_persona'].value_counts(dropna=False))

### ¿A quién describen los archivos de personas?

Al comparar los tres archivos comprobamos que `sexo` y `condicion_persona` coinciden entre las tablas de edad y sexo. También encontramos que `total` coincide exactamente con `personas_lesionadas` de `hechos_ssc` en todos los folios. **No debemos sumar esos campos entre archivos** porque repetiríamos los mismos conteos.

El problema se encuentra en que cada folio tiene sólo un valor de `edad`, `sexo` y `tipo_persona`, incluso cuando reporta varias personas afectadas.Tampoco `condicion_persona` representa necesariamente todas las consecuencias del evento ya que encontramos incidentes con lesionados y fallecidos a la vez, además de ocho registros sin víctimas contabilizadas.

Por todo lo anterior empezaremos investigando los **25,983 incidentes con una persona lesionada y ningún fallecimiento**. En ellos hay 18,445 registros masculinos, 7,538 femeninos y 1,101 edades faltantes. Esta selección reduce la ambigüedad, aunque se ve limitada por cómo se asignaron los atributos en la fuente de los datos.

In [ ]:
# Número de casos por tipo de participante y sexo
tabla_sexo_persona = pd.crosstab(casos_individuales['tipo_persona'], casos_individuales['sexo'])

tabla_sexo_persona_pct = (tabla_sexo_persona.div(tabla_sexo_persona.sum(axis=1), axis=0) * 100)

print('Frecuencias absolutas:')
print(tabla_sexo_persona)

print('\nPorcentajes por tipo de participante:')
print(tabla_sexo_persona_pct.round(2))

Primero veremos cómo se distribuye el sexo registrado **dentro de cada tipo de participante**.

In [ ]:
datos_plot = tabla_sexo_persona_pct.sort_values('hombre', ascending=True)

fig, ax = plt.subplots(figsize=(8, 4.5))

# Barras apiladas al 100%
datos_plot[['hombre', 'mujer']].plot(kind='barh', stacked=True, ax=ax, color=['#477DB3', '#D887AC'], width=0.7)

# Etiquetas de porcentaje dentro de cada segmento
for contenedor in ax.containers:
    ax.bar_label(
        contenedor,
        fmt='%.1f%%',
        label_type='center',
        color='white',
        fontsize=10,
        fontweight='bold'
    )

ax.set_title('Distribución por sexo y tipo de participante (Personas Lesionadas) - CDMX 2024')
ax.set_xlabel('Porcentaje de registros')
ax.set_ylabel('Tipo de participante')
ax.set_xlim(0, 100)
ax.legend(title='Sexo', loc='lower right')

plt.tight_layout()
plt.show()

### Sexo según el tipo de participante

Podemos ver que entre los motociclistas lesionados de nuestra selección, **84.4% son hombres**, también predominan entre ciclistas (74.3%) y conductores (71.6%). En pasajeros ocurre lo contrario: **74.3% son mujeres** y entre peatones la diferencia es menor.

Estas cifras describen a los participantes registrados en incidentes con **una sola persona lesionada y ningún fallecimiento**.

Ahora invertiremos la pregunta. En lugar de preguntar cuántos hombres y mujeres hay entre los grupos, veremos **qué tipo de participante aparece dentro de cada sexo**.

In [ ]:
# Distribución de tipos de participante dentro de cada sexo
tabla_persona_sexo_pct = (tabla_sexo_persona.div(tabla_sexo_persona.sum(axis=0), axis=1) * 100)

print(tabla_persona_sexo_pct.round(2))

# Comprobamos que cada columna sume 100%
print("\nCheck integridad:", tabla_persona_sexo_pct.sum(axis=0))

In [ ]:
# Transponemos para que las barras sean hombre y mujer
datos_plot = tabla_persona_sexo_pct.T

# Colores por tipo de participante
colores_graph_mult = [
    '#397C8C',  # Ciclista
    '#7184AE',  # Conductor
    '#C56C54',  # Motociclista
    '#D9AC62',  # Pasajero
    '#8D83A5'   # Peatón
]

fig, ax = plt.subplots(figsize=(8, 6))

datos_plot.plot(kind='bar', stacked=True, ax=ax, color=colores_graph_mult, width=0.55)

# Añadir etiquetas dentro de cada segmento
for i, sexo in enumerate(datos_plot.index):
    acumulado = 0
    for participante in datos_plot.columns:
        valor = datos_plot.loc[sexo, participante]

        if valor > 0:
            ax.text(
                i,                              # posición x de la barra
                acumulado + valor / 2,         # centro del segmento
                f'{valor:.1f}%',
                ha='center',
                va='center',
                fontsize=10,
                color='white',
                fontweight='bold'
            )
        acumulado += valor

ax.set_title('Composición de participantes lesionados por sexo - CDMX 2024')
ax.set_xlabel('Sexo')
ax.set_ylabel('Porcentaje de registros')
ax.set_ylim(0, 100)
ax.tick_params(axis='x', rotation=0)

ax.legend(
    title='Tipo de participante',
    bbox_to_anchor=(1.02, 1),
    loc='upper left'
)

plt.tight_layout()
plt.show()

### Tipo de participante dentro de cada sexo

Los motociclistas representan **62.7% de los registros masculinos**. Entre las mujeres, en cambio, la distribución es más pareja: motociclistas (28.4%), pasajeras (27.5%) y peatonas (25.1%).

Cambiar el denominador cambia la lectura brindando conteos relativos, es decir, que 84.4% de los motociclistas de la selección sean hombres no significa que 84.4% de los hombres registrados sean motociclistas.

Es natural preguntarnos ahora si: ¿Encontraremos un perfil similar entre los incidentes con fallecimientos?

In [ ]:
# Incidentes con una persona fallecida y ninguna lesionada
casos_fallecidos = comparacion_totales.loc[(comparacion_totales['personas_fallecidas'] == 1) & (comparacion_totales['personas_lesionadas'] == 0)].copy()

print('Incidentes seleccionados:', len(casos_fallecidos))

print('\nCondición registrada:')
print(casos_fallecidos['condicion_persona'].value_counts())

print('\nDistribución del sexo:')
print(casos_fallecidos['sexo'].value_counts(dropna=False))

print('\nDistribución del tipo de persona:')
print(casos_fallecidos['tipo_persona'].value_counts(dropna=False))

In [ ]:
# Distribución del número de fallecidos por incidente
distribucion_fallecidos = comparacion_totales.loc[comparacion_totales['personas_fallecidas'] > 0, 'personas_fallecidas'].value_counts().sort_index()

print(distribucion_fallecidos)

# Cuántos fallecimientos cubre nuestro subconjunto inicial
print('\nFallecimientos totales:', comparacion_totales['personas_fallecidas'].sum())

print('Fallecimientos en la selección individual:', len(casos_fallecidos))

### Participantes registrados en incidentes con un fallecimiento

Para reducir la ambigüedad seleccionamos los incidentes con **exactamente una persona fallecida y ninguna lesionada**. Son 389 casos, con una mayor presencia de motociclistas (46.0%) y peatones (36.2%).

La selección excluye eventos con lesionados adicionales o más de una persona fallecida. Por lo tanto, no describe necesariamente a las **533 víctimas mortales** de todo el año.

In [ ]:
# Revisamos la condición registrada
print(casos_fallecidos['condicion_persona'].value_counts())

# Distribución del sexo
print('\n', casos_fallecidos['sexo'].value_counts(dropna=False))

# Distribución del tipo de participante
print('\n', casos_fallecidos['tipo_persona'].value_counts(dropna=False))

In [ ]:
resumen_fallecidos = casos_fallecidos['tipo_persona'].value_counts().rename_axis('tipo_persona').reset_index(name='registros')

resumen_fallecidos['porcentaje'] = resumen_fallecidos['registros'] / resumen_fallecidos['registros'].sum() * 100

fig, ax = plt.subplots(figsize=(8, 4))

sns.barplot(
    data=resumen_fallecidos,
    x='porcentaje',
    y='tipo_persona',
    hue='tipo_persona',
    palette=colores_graph_mult,
    legend=False,
    ax=ax
)

for contenedor in ax.containers:
    ax.bar_label(contenedor, fmt='%.1f%%', padding=4)

ax.set_xlim(0, resumen_fallecidos['porcentaje'].max() * 1.2)
ax.set_title('Tipo de participante en la selección de fallecidos - CDMX 2024')
ax.set_xlabel('Porcentaje de registros seleccionados')
ax.set_ylabel('Tipo de participante')

plt.tight_layout()
plt.show()

Hay algo extraño ya que los 389 registros de esta selección aparecen con sexo masculino. Revisemos todos los incidentes fatales antes de interpretar esa cifra.

In [ ]:
# Seleccionamos todos los incidentes con al menos un fallecimiento
incidentes_fatales = comparacion_totales.loc[comparacion_totales['personas_fallecidas'] > 0]

# Sexo registrado según la condición de la persona
tabla_sexo_fatal = pd.crosstab(incidentes_fatales['condicion_persona'], incidentes_fatales['sexo'], margins=True)

display(tabla_sexo_fatal)

# Revisamos también cuántos fallecidos registra cada folio
display(pd.crosstab(incidentes_fatales['personas_fallecidas'], incidentes_fatales['sexo'])
)

### Sexo registrado en los incidentes fatales

Al ampliar la revisión a los **512 incidentes con al menos un fallecimiento**, encontramos sexo masculino en 90.6% de los folios, femenino en 7.2% y desconocido en 2.1%.

Esto confirma que la selección anterior no representa todos los incidentes fatales. Además, recordemos que tenemos **un solo sexo por folio**, incluso en eventos con varias víctimas.

In [ ]:
# Sexo registrado en incidentes con al menos un fallecimiento
resumen_sexo_fatal = (incidentes_fatales['sexo'].value_counts().rename_axis('sexo').reset_index(name='incidentes'))

resumen_sexo_fatal['porcentaje'] = (resumen_sexo_fatal['incidentes'] / resumen_sexo_fatal['incidentes'].sum() * 100)

colores_sexo = {
    'hombre': '#477DB3',
    'mujer': '#D887AC',
    'desconocido': '#A0A0A0'
}

fig, ax = plt.subplots(figsize=(8, 3.5))

sns.barplot(
    data=resumen_sexo_fatal,
    x='porcentaje',
    y='sexo',
    hue='sexo',
    palette=colores_sexo,
    legend=False,
    ax=ax
)

for contenedor in ax.containers:
    ax.bar_label(
        contenedor,
        fmt='%.1f%%',
        padding=4
    )

ax.set_xlim(0, 100)
ax.set_title('Sexo registrado en incidentes fatales - CDMX 2024')
ax.set_xlabel('Porcentaje de incidentes con fallecimientos')
ax.set_ylabel('Sexo registrado')

plt.tight_layout()
plt.show()

**Lo que aprendimos:** los perfiles registrados cambian bastante según el tipo de participante, pero también según la selección que hagamos. Por eso, al pasar a las edades mantendremos el subconjunto utilizado de incidentes con una sola persona lesionada y ningún fallecimiento para evitar introducir ambigüedad a nuestra exploración.

## ¿Qué edades tienen los participantes lesionados?

Empezaremos revisando la distribución general y luego compararemos el tipo de participante (por grupo de vehiculo).

In [ ]:
# Resumen de edades de los casos individuales
print('Registros:', len(casos_individuales))
print('Edades disponibles:', casos_individuales['edad'].notna().sum())
print('Edades faltantes:', casos_individuales['edad'].isna().sum())
print('\n', casos_individuales['edad'].describe())

De los **25,983** casos seleccionados, **24,882** tienen edad disponible. La mediana es **30 años** y el 50% central se encuentra entre los **24 y 41 años**. Veamos la forma de esa distribución antes de separarla por participantes.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

sns.histplot(data=casos_individuales, x='edad', bins=range(0, 101, 5), color='#397C8C', edgecolor='white', ax=ax)

ax.axvline(casos_individuales['edad'].median(), color='#C56C54', linestyle='--', linewidth=2, label='Mediana')

ax.axvline(casos_individuales['edad'].mean(), color='#D9AC62', linestyle=':', linewidth=2, label='Media')

ax.set_title('Distribución de edades registradas - CDMX 2024')
ax.set_xlabel('Edad (años)')
ax.set_ylabel('Número de registros')
ax.legend()
plt.tight_layout()
plt.show()

La distribución se concentra entre los **20 y 40 años**, con una cola hacia edades mayores. Dado el peso de las motocicletas en los registros, vale la pena revisar cada grupo de participantes por separado.

In [ ]:
# Resumen de edades por tipo de participante
resumen_edades = (casos_individuales.groupby('tipo_persona')['edad'].agg(['count', 'mean', 'median', 'std', 'min', 'max']).sort_values('median'))

print(resumen_edades.round(2))

In [ ]:
orden_participantes = [
    'MOTOCICLISTA',
    'PASAJERO',
    'CICLISTA',
    'CONDUCTOR',
    'PEATON'
]

fig, ax = plt.subplots(figsize=(8, 5))

sns.boxplot(
    data=casos_individuales,
    x='edad',
    y='tipo_persona',
    order=orden_participantes,
    hue='tipo_persona',
    palette=colores_graph_mult,
    dodge=False,
    legend=False,
    showfliers=False,
    ax=ax
)

ax.set_title('Distribución de edades por tipo de participante - CDMX 2024')
ax.set_xlabel('Edad (años)')
ax.set_ylabel('Tipo de participante')

plt.tight_layout()
plt.show()

In [ ]:
#Veamos con mas detalle con un violinplot

fig, ax = plt.subplots(figsize=(8, 5))

sns.violinplot(
    data=casos_individuales,
    x='edad',
    y='tipo_persona',
    order=orden_participantes,
    hue='tipo_persona',
    palette=colores_graph_mult,
    inner='quart',
    density_norm='width',
    cut=0,
    legend=False,
    ax=ax
)

ax.set_title('Densidad de edades por tipo de participante - CDMX 2024')
ax.set_xlabel('Edad (años)')
ax.set_ylabel('Tipo de participante')

plt.tight_layout()
plt.show()

### Edad según el tipo de participante

Los **motociclistas** tienen la mediana más baja (28 años) y una distribución concentrada. Los **peatones** presentan la mediana más alta (39 años) y una dispersión mayor. Entre ambos aparecen pasajeros (31), ciclistas (32) y conductores (36).

Los diagramas de caja nos muestran las medianas y la variabilidad, mientras que los de violín permiten observar dónde se acumulan las edades. Seguimos trabajando solo con los casos individuales que tienen edad registrada.

Ahora veremos si las distribuciones de edad cambian al separar hombres y mujeres dentro de cada tipo de participante.

In [ ]:
resumen_edad_sexo = (casos_individuales.groupby(['tipo_persona', 'sexo'])['edad'].agg(['count', 'median', 'mean', 'std']).round(2))

print(resumen_edad_sexo)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

sns.boxplot(
    data=casos_individuales,
    x='edad',
    y='tipo_persona',
    hue='sexo',
    order=orden_participantes,
    hue_order=['hombre', 'mujer'],
    palette=colores_sexo,
    showfliers=False,
    ax=ax
)

ax.set_title('Distribución de edades por participante y sexo - CDMX 2024')
ax.set_xlabel('Edad (años)')
ax.set_ylabel('Tipo de participante')
ax.legend(title='Sexo')
plt.tight_layout()
plt.show()

### Edad, sexo y tipo de participante

Las medianas de motociclistas son prácticamente iguales entre hombres y mujeres (**28 años** en ambos casos). En pasajeros sí aparece una diferencia: **27 años** entre hombres y **32** entre mujeres. Peatones y conductores muestran otras variaciones menores.

Estas comparaciones describen los registros disponibles y no deben de interpretarse como una distribución general de todas las personas que circulan por la ciudad.

In [ ]:
# Registros con edad menor de cinco años
casos_menores_5 = casos_individuales.loc[casos_individuales['edad'] < 5].copy()

print('Casos identificados:', len(casos_menores_5))

print('\nEdades registradas:')
print(casos_menores_5['edad'].value_counts().sort_index())

print('\nTipo de participante:')
print(casos_menores_5['tipo_persona'].value_counts())

print('\nSexo registrado:')
print(casos_menores_5['sexo'].value_counts())

Revisaremos ahora los registros con edades inferiores a cinco años. La mayoría corresponde a pasajeros o peatones, pero también aparecen menores clasificados como motociclistas y un caso como conductor. Revisemos cómo fueron registrados esos eventos.

In [ ]:
# Casos de menores de 5 años clasificados como motociclistas o conductores
casos_inusuales = casos_menores_5.loc[casos_menores_5['tipo_persona'].isin(['MOTOCICLISTA', 'CONDUCTOR'])].reset_index()

detalle_menores = casos_inusuales[['folio', 'edad', 'sexo', 'tipo_persona']].merge(
    dfs['hechos_ssc'][['folio', 'fecha_evento', 'tipo_evento', 'alcaldia_utilizada', 'punto_1', 'punto_2']],
    on='folio', validate='one_to_one'
).merge(dfs['vehiculos'][['folio', 'tipo_vehiculo']], on='folio', validate='one_to_one')

display(detalle_menores)

In [ ]:
# Relacionamos los menores de cinco años con el tipo de evento
menores_eventos = casos_menores_5.reset_index()[['folio', 'edad', 'tipo_persona']].merge(
    dfs['hechos_ssc'][['folio', 'tipo_evento']], on='folio', validate='one_to_one')

tabla_menores_eventos = pd.crosstab(menores_eventos['tipo_persona'], menores_eventos['tipo_evento'])

print(tabla_menores_eventos)

# Comprobamos que no perdimos registros
print('Total de casos:', tabla_menores_eventos.to_numpy().sum())

In [ ]:
datos_plot = tabla_menores_eventos.loc[tabla_menores_eventos.sum(axis=1).sort_values().index]

colores_eventos = {
    'ATROPELLADO': '#C56C54',
    'CAIDA DE CICLISTA': '#397C8C',
    'CAIDA DE PASAJERO': '#8D83A5',
    'CHOQUE': '#7184AE',
    'DERRAPADO': '#D9AC62'
}

fig, ax = plt.subplots(figsize=(8, 4.5))

datos_plot.plot(
    kind='barh',
    stacked=True,
    ax=ax,
    color=[colores_eventos[c] for c in datos_plot.columns],
    width=0.65
)

# Etiquetas dentro de segmentos suficientemente grandes
for contenedor in ax.containers:
    etiquetas = [
        str(int(v)) if v >= 3 else ''
        for v in contenedor.datavalues
    ]
    ax.bar_label(
        contenedor,
        labels=etiquetas,
        label_type='center',
        fontsize=10,
        color='white',
        fontweight='bold'
    )

ax.set_title('Incidentes con menores de 5 años lesionados - CDMX 2024')
ax.set_xlabel('Número de incidentes')
ax.set_ylabel('Tipo de participante')
ax.legend(title='Tipo de evento', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

### Menores de cinco años en la selección

Encontramos **83 incidentes** con una persona lesionada menor de cinco años y edad disponible: 46 registrados como pasajeros y 29 como peatones. En esos grupos predominan choques y atropellamientos, respectivamente.

También aparecen **siete menores** clasificados como motociclistas y **uno de dos años como conductor**. La clasificación no permite asegurar qué papel tuvieron realmente; en particular, no interpretamos que un niño de dos años condujera el taxi asociado a ese folio del registro.

La selección no representa todos los menores lesionados en 2024. Antes de intentar un perfil de víctimas mortales, revisaremos la calidad de los datos de edad en los incidentes fatales.

In [ ]:
# Incidentes con exactamente una persona fallecida
casos_fatales_individuales = comparacion_totales.loc[comparacion_totales['personas_fallecidas'] == 1].copy()

print('Incidentes seleccionados:', len(casos_fatales_individuales))

print('\nCondición registrada:')
print(casos_fatales_individuales['condicion_persona'].value_counts())

print('\nSexo registrado:')
print(casos_fatales_individuales['sexo'].value_counts(dropna=False))

print('\nEdades disponibles:')
print(casos_fatales_individuales['edad'].describe())

La edad mediana registrada en los incidentes con un fallecimiento es de **dos años**, un resultado inesperado. Revisaremos los valores y cuántos faltan.

In [ ]:
print('Edades registradas:')
print(casos_fatales_individuales['edad'].value_counts(dropna=False).sort_index())

print('\nDisponibilidad de edad por sexo:')
print(
    pd.crosstab(
        casos_fatales_individuales['sexo'],
        casos_fatales_individuales['edad'].notna(),
        margins=True
    )
)

print('\nDisponibilidad de edad por tipo de participante:')
print(
    pd.crosstab(
        casos_fatales_individuales['tipo_persona'],
        casos_fatales_individuales['edad'].notna(),
        margins=True
    )
)

Los valores parecen extraños incluso antes de analizarlos, asi que compararemos con el CSV original para comprobar si no introdujimos un error durante la limpieza.

In [ ]:
# Recuperamos las edades originales (sin limpieza) desde el CSV
ruta_original = ('/content/drive/MyDrive/ProyectosPython/cdmx-road-safety-intelligence/data/raw/lesionados_edadcsv.csv')

edad_original = pd.read_csv(ruta_original, dtype={'edad': 'string'}, low_memory=False)

# Comparamos la edad original con la procesada
comparacion_edades = (
    casos_fatales_individuales[['edad', 'sexo', 'tipo_persona']].reset_index()
    .merge(edad_original[['folio', 'edad']], on='folio', suffixes=('_procesada', '_original'), validate='one_to_one')
)

display(comparacion_edades[['sexo', 'tipo_persona', 'edad_original', 'edad_procesada']].value_counts(dropna=False).head(25))

### Edad en incidentes fatales: una limitación de la fuente

Entre los **495 incidentes con exactamente una persona fallecida**, solo **108 (21.8%)** tienen una edad disponible. De ellas, **103** están entre cero y seis años. Además, los faltantes no se distribuyen de manera uniforme entre los grupos registrados.

Confirmamos que los valores ya estaban en el CSV original y no fueron introducidos durante la limpieza. Con esta cobertura y distribución no es razonable construir un perfil de edades de personas fallecidas. **Asi que conservamos los conteos de fallecimientos, pero dejamos esta variable fuera de ese análisis.**

## ¿A qué horas ocurren más incidentes?

Ya vimos variaciones mensuales. Ahora revisaremos la hora y el día de la semana para buscar patrones más específicos respecto a la distribución de los choques a lo largo del día.

In [ ]:
hechos_por_hora = dfs['hechos_ssc']['hora_evento'].map(lambda hora: hora.hour).value_counts().reindex(range(24), fill_value=0)

print(hechos_por_hora)

El máximo aparece alrededor de las **08:00**, seguido de actividad elevada durante la tarde y un repunte hacia las **19:00**.

### ¿Cambiará la distribución entre días laborales y fines de semana?

In [ ]:
# Visualizamos distribución por horas

fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(hechos_por_hora.index, hechos_por_hora.values, color='#397C8C', marker='o', linewidth=2)

ax.fill_between(hechos_por_hora.index, hechos_por_hora.values, color='#397C8C', alpha=0.12)

ax.set_title('Distribución horaria de incidentes viales - CDMX 2024')
ax.set_xlabel('Hora del día')
ax.set_ylabel('Número de incidentes')
ax.set_xticks(range(0, 24, 2))
ax.set_xlim(0, 23)
ax.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.show()

Crucemos las horas con el día de la semana.

In [ ]:
df_temporal = dfs['hechos_ssc'].copy()

df_temporal['hora'] = df_temporal['hora_evento'].map(lambda hora: hora.hour)

orden_dias = ['lunes', 'martes', 'miércoles', 'jueves', 'viernes', 'sábado', 'domingo']

# Contamos incidentes por día de la semana y hora
tabla_dia_hora = pd.crosstab(df_temporal['dia'], df_temporal['hora'])

tabla_dia_hora = tabla_dia_hora.reindex(index=orden_dias, columns=range(24), fill_value=0)

with pd.option_context('display.max_columns', None, 'display.width', 1000):
    print(tabla_dia_hora)

En los días laborales destaca la mañana, mientras que durante sábado y domingo la madrugada tiene más presencia.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))

sns.heatmap(
    tabla_dia_hora,
    cmap='YlOrRd',
    annot=False,
    linewidths=0.3,
    cbar_kws={'label': 'Número de incidentes'},
    ax=ax
)

ax.set_title('Incidentes viales por día y hora - CDMX 2024')
ax.set_xlabel('Hora del día')
ax.set_ylabel('Día de la semana')
ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.show()

Ahora introducimos un conteo relativo expresando cada hora como porcentaje de los incidentes registrados **en ese día de la semana** para ver si el patrón se mantiene al normalizar dentro de cada día.

In [ ]:
# Porcentaje de incidentes de cada día que ocurre en cada hora
tabla_dia_hora_pct = (tabla_dia_hora.div(tabla_dia_hora.sum(axis=1), axis=0) * 100)

fig, ax = plt.subplots(figsize=(12, 4.5))

sns.heatmap(
    tabla_dia_hora_pct,
    cmap='YlOrRd',
    annot=False,
    linewidths=0.3,
    cbar_kws={'label': 'Porcentaje de incidentes del día'},
    ax=ax
)

ax.set_title('Distribución horaria relativa por día - CDMX 2024')
ax.set_xlabel('Hora del día')
ax.set_ylabel('Día de la semana')
ax.tick_params(axis='y', rotation=0)
plt.tight_layout()
plt.show()

### Días de la semana y horarios

El heatmap porcentual conserva prácticamente la misma forma que el absoluto, así que no añade grandes diferencias. De lunes a viernes destacan las **07:00–10:00**, con máximos cerca de las 08:00. Los fines de semana muestran más actividad en la madrugada, especialmente entre las 00:00 y las 04:00.

Ahora investigaremos si **¿los incidentes fatales siguen esos mismos horarios?**

In [ ]:
# Identificamos incidentes según sus consecuencias
df_temporal['con_lesionados'] = df_temporal['personas_lesionadas'] > 0

df_temporal['con_fallecidos'] = df_temporal['personas_fallecidas'] > 0

# Contamos incidentes por hora
consecuencias_por_hora = df_temporal.groupby('hora').agg(
    incidentes=('folio', 'count'),
    con_lesionados=('con_lesionados', 'sum'),
    con_fallecidos=('con_fallecidos', 'sum')
)

# Porcentaje de incidentes de cada hora con fallecimientos
consecuencias_por_hora['pct_fatales'] = consecuencias_por_hora['con_fallecidos'] / consecuencias_por_hora['incidentes'] * 100
print(consecuencias_por_hora.round(2))

Comparemos los casos con lesionados, los casos con fallecimientos y la proporción fatal en cada hora.

In [ ]:
fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(9, 8), sharex=True, gridspec_kw={'height_ratios': [1, 1, 1.2]})

hora = consecuencias_por_hora.index

metricas = [
    ('con_lesionados', 'Incidentes con lesionados', '#397C8C'),
    ('con_fallecidos', 'Incidentes con fallecimientos', '#C56C54'),
    ('pct_fatales', 'Proporción de incidentes fatales (%)', '#D9AC62')
]

for ax, (columna, etiqueta, color) in zip(axes, metricas):
    ax.plot(hora, consecuencias_por_hora[columna], color=color, marker='o', markersize=4, linewidth=2)

    ax.fill_between(hora, consecuencias_por_hora[columna], color=color, alpha=0.12)

    ax.set_ylabel(etiqueta)
    ax.grid(axis='y', alpha=0.2)

axes[0].set_title('Distribución horaria según consecuencias - CDMX 2024')
axes[-1].set_xlabel('Hora del día')
axes[-1].set_xticks(range(24))
axes[-1].set_xlim(0, 23)

plt.tight_layout()
plt.show()

### Consecuencias según la hora

De esta grafica podríamos ver que, por ejemplo, a las **08:00** se registraron **1,936 incidentes**, de los cuales **17** tuvieron al menos un fallecimiento (**0.88%**). A las **04:00** hubo solo **501**, pero **38** fueron fatales (**7.58%**).

Así, la proporción fatal a las 04:00 fue aproximadamente **8.6 veces** la de las 08:00. Mostrando que los horarios con más incidentes no coinciden necesariamente con los de mayor proporción fatal. Además, los conteos de incidentes con lesionados y con fallecimientos pueden superponerse cuando un mismo evento tiene ambas consecuencias.

Por otro lado, esto coincide con la idea popular de que "la madrugada es más peligrosa", algunas razones que creemos posibles son: al haber menos tráfico se circula más rápido, los conductores pueden estar cansados y/o bajo el efecto de bebidas alcoholicas o estupefacientes o que no se respetan los semáforos.

¿Esa concentración de incidentes fatales en la madrugada cambia según el día de la semana?

In [ ]:
# Incidentes con fallecimientos por día y hora
tabla_fatales_dia_hora = pd.crosstab(df_temporal['dia'], df_temporal['hora'], values=df_temporal['con_fallecidos'].astype(int), aggfunc='sum')

tabla_fatales_dia_hora = tabla_fatales_dia_hora.reindex(index=orden_dias, columns=range(24), fill_value=0).fillna(0).astype(int)

with pd.option_context('display.max_columns', None, 'display.width', 1000):
    print(tabla_fatales_dia_hora)

print('\nTotal de incidentes fatales:', tabla_fatales_dia_hora.to_numpy().sum())

En los conteos aparecen máximos como **domingo a las 04:00 (13 casos)** y **sábado a las 05:00 (11)**. Pero también es importante comparar contra el total de incidentes de cada combinación de día y hora.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))

sns.heatmap(
    tabla_fatales_dia_hora,
    cmap='OrRd',
    annot=False,
    linewidths=0.3,
    cbar_kws={'label': 'Número de incidentes fatales'},
    ax=ax
)

ax.set_title('Incidentes fatales por día y hora - CDMX 2024')
ax.set_xlabel('Hora del día')
ax.set_ylabel('Día de la semana')
ax.tick_params(axis='x', rotation=0)
ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.show()

### Incidentes fatales por día y hora

El heatmap muestra **cuántos incidentes con fallecimientos** se registraron en cada combinación de día y hora. Algunas de las celdas más intensas aparecen en madrugadas de sábado y domingo, sugiriendo consistencia con la grafica anterior.

El conteo absoluto no nos dice si ese patrón se mantiene cuando consideramos cuántos incidentes hubo en total por día. Por eso repetiremos la visualización en porcentajes.

In [ ]:
# Porcentaje de incidentes fatales dentro de cada día y hora
tabla_pct_fatales = tabla_fatales_dia_hora.div(tabla_dia_hora.where(tabla_dia_hora.ne(0))) * 100

fig, ax = plt.subplots(figsize=(10, 4.5))
sns.heatmap(tabla_pct_fatales, cmap='YlOrRd', linewidths=0.3, vmin=0,
            cbar_kws={'label': 'Incidentes fatales (%)'}, ax=ax)

ax.set_title('Proporción de incidentes fatales por día y hora - CDMX 2024')
ax.set_xlabel('Hora del día')
ax.set_ylabel('Día de la semana')
ax.tick_params(axis='x', rotation=0)
ax.tick_params(axis='y', rotation=0)
plt.tight_layout()
plt.show()

### Proporción fatal por día y hora

Ahora cada celda representa el **porcentaje de incidentes con al menos un fallecimiento** dentro de ese día y hora. La madrugada vuelve a destacar en distintas partes de la semana.

Los porcentajes de celdas con pocos registros pueden variar mucho por lo que, para comprobar si el contraste general se mantiene, agruparemos las horas en bloques más amplios.

Ambos mapas de calor apuntan a una mayor presencia relativa de incidentes fatales durante la madrugada. Veamos si se trata especialmente de los fines de semana o del horario en general.

### ¿Importa más la madrugada o el fin de semana?

Agruparemos las horas en madrugada (00:00–05:59), mañana (06:00–11:59), tarde (12:00–17:59) y noche (18:00–23:59). Después compararemos días entre semana y fines de semana.

In [ ]:
# Definimos bloques horarios
def clasificar_horario(hora):
    if 0 <= hora < 6:
        return 'Madrugada'
    elif 6 <= hora < 12:
        return 'Mañana'
    elif 12 <= hora < 18:
        return 'Tarde'
    else:
        return 'Noche'

df_temporal['bloque_horario'] = df_temporal['hora'].map(clasificar_horario)

# Diferenciamos días laborales y fines de semana
df_temporal['tipo_dia'] = df_temporal['dia'].map(lambda dia: 'Fin de semana' if dia in ['sábado', 'domingo'] else 'Entre semana')

# Resumen de frecuencia y consecuencias
resumen_bloques = (
    df_temporal.groupby(['tipo_dia', 'bloque_horario'])
    .agg(
        incidentes=('folio', 'count'),
        con_lesionados=('con_lesionados', 'sum'),
        con_fallecidos=('con_fallecidos', 'sum')
    )
)

# Porcentajes respecto al total de incidentes de cada combinación
resumen_bloques['pct_lesionados'] = resumen_bloques['con_lesionados'] / resumen_bloques['incidentes'] * 100

resumen_bloques['pct_fatales'] = resumen_bloques['con_fallecidos'] / resumen_bloques['incidentes'] * 100

display(resumen_bloques.round(2))

La proporción de incidentes con lesionados está cerca del 100% en casi todos los bloques. Para apreciar diferencias más útiles, compararemos **personas lesionadas por 100 incidentes** y **porcentaje de incidentes fatales**.

In [ ]:
resumen_bloques = (
    df_temporal.groupby(['tipo_dia', 'bloque_horario'])
    .agg(
        incidentes=('folio', 'count'),
        personas_lesionadas=('personas_lesionadas', 'sum'),
        personas_fallecidas=('personas_fallecidas', 'sum'),
        incidentes_fatales=('con_fallecidos', 'sum')
    )
)

# Personas lesionadas por cada 100 incidentes
resumen_bloques['lesionados_por_100'] = resumen_bloques['personas_lesionadas'] / resumen_bloques['incidentes'] * 100

# Porcentaje de incidentes con al menos un fallecimiento
resumen_bloques['pct_fatales'] = resumen_bloques['incidentes_fatales'] / resumen_bloques['incidentes'] * 100

display(resumen_bloques.round(2))

In [ ]:
# Creamos visualizaciones
orden_bloques = ['Madrugada', 'Mañana', 'Tarde', 'Noche']

colores_tipo_dia = {
    'Entre semana': '#7184AE',
    'Fin de semana': '#C56C54'
}

tabla_bloques_plot = resumen_bloques.reset_index()

fig, ax = plt.subplots(figsize=(8, 4.5))

sns.barplot(
    data=tabla_bloques_plot,
    x='bloque_horario',
    y='lesionados_por_100',
    hue='tipo_dia',
    order=orden_bloques,
    hue_order=['Entre semana', 'Fin de semana'],
    palette=colores_tipo_dia,
    ax=ax
)

for contenedor in ax.containers:
    ax.bar_label(contenedor, fmt='%.1f', padding=3, fontsize=9)

ax.set_title('Personas lesionadas por bloque horario - CDMX 2024')
ax.set_xlabel('Bloque horario')
ax.set_ylabel('Personas lesionadas por cada 100 incidentes')
ax.legend(title='Tipo de día', loc='upper left', bbox_to_anchor=(1.02, 1))
ax.set_ylim(0, 140)
ax.grid(axis='y', alpha=0.2)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

sns.barplot(
    data=tabla_bloques_plot,
    x='bloque_horario',
    y='pct_fatales',
    hue='tipo_dia',
    order=orden_bloques,
    hue_order=['Entre semana', 'Fin de semana'],
    palette=colores_tipo_dia,
    ax=ax
)

for contenedor in ax.containers:
    ax.bar_label(contenedor, fmt='%.2f%%', padding=3,fontsize=9    )

ax.set_title('Proporción de incidentes fatales por bloque horario - CDMX 2024')
ax.set_xlabel('Bloque horario')
ax.set_ylabel('Incidentes con fallecimientos (%)')

ax.legend(title='Tipo de día', loc='upper left', bbox_to_anchor=(1.02, 1))
ax.set_ylim(0, 5.2)
ax.grid(axis='y', alpha=0.2)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

### Consecuencias por bloque horario

En los cuatro bloques hay más personas lesionadas por 100 incidentes durante el fin de semana. Destaca la **noche de sábado y domingo**, con **123.12**.

En cambio, la proporción de incidentes fatales es más alta en la **madrugada**, tanto entre semana (**4.39%**) como en fin de semana (**4.47%**). Son valores casi iguales: la diferencia no parece concentrarse exclusivamente en sábados y domingos.

El número de lesionados por incidente y la proporción fatal revelan facetas distintas de las consecuencias registradas.

## ¿Cómo cambian los incidentes día a día?

Ahora extenderemos el análisis a los **366 días de 2024**. Construiremos dos series independientes: incidentes con lesionados e incidentes con al menos un fallecimiento.

In [ ]:
# Calendario completo de 2024 (año bisiesto)
calendario_2024 = pd.date_range(start='2024-01-01', end='2024-12-31', freq='D')

# Conteo diario de incidentes según sus consecuencias
series_diarias = (
    df_temporal.groupby('fecha_evento')
    .agg(
        con_lesionados=('con_lesionados', 'sum'),
        con_fallecidos=('con_fallecidos', 'sum')
    )
    .reindex(calendario_2024, fill_value=0)
)

series_diarias.index.name = 'fecha'

print(series_diarias.head(10))
print('Días del calendario:', len(series_diarias))
print('Incidentes con lesionados:', series_diarias['con_lesionados'].sum())
print('Incidentes fatales:', series_diarias['con_fallecidos'].sum())
print('\nResumen estadístico:')
print(series_diarias.describe().round(2))
print('\nDías con más incidentes fatales:')
print(series_diarias.nlargest(10, 'con_fallecidos'))

In [ ]:
import matplotlib.dates as mdates

fig, ax = plt.subplots(figsize=(9, 4.5))

ax.plot(
    series_diarias.index,
    series_diarias['con_lesionados'],
    color='#397C8C',
    linewidth=1.5
)

ax.fill_between(
    series_diarias.index,
    series_diarias['con_lesionados'],
    color='#397C8C',
    alpha=0.10
)

ax.axhline(
    series_diarias['con_lesionados'].mean(),
    color='#D9AC62',
    linestyle='--',
    linewidth=1.5,
    label='Promedio diario'
)

ax.set_title('Evolución diaria de incidentes con lesionados - CDMX 2024')
ax.set_xlabel('Fecha')
ax.set_ylabel('Incidentes con lesionados')

ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))

ax.set_xlim(series_diarias.index.min(), series_diarias.index.max())
ax.set_ylim(bottom=0)
ax.grid(axis='y', alpha=0.2)
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))

ax.plot(
    series_diarias.index,
    series_diarias['con_fallecidos'],
    color='#C56C54',
    linewidth=1.5
)

ax.fill_between(
    series_diarias.index,
    series_diarias['con_fallecidos'],
    color='#C56C54',
    alpha=0.12
)

ax.axhline(
    series_diarias['con_fallecidos'].mean(),
    color='#D9AC62',
    linestyle='--',
    linewidth=1.5,
    label='Promedio diario'
)

ax.set_title('Evolución diaria de incidentes fatales - CDMX 2024')
ax.set_xlabel('Fecha')
ax.set_ylabel('Incidentes con fallecimientos')

ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))

ax.set_xlim(series_diarias.index.min(), series_diarias.index.max())
ax.set_ylim(bottom=0)
ax.set_yticks(range(0, 9))
ax.grid(axis='y', alpha=0.2)
ax.legend()

plt.tight_layout()
plt.show()

Las series cuentan su propia historia, los incidentes con lesionados parecen disminuir en julio y parte de agosto, mientras que los fatales fluctúan más y presentan picos aislados. El máximo fatal registrado fue el **9 de junio, con siete incidentes**. Veamos las fechas extremas antes de buscar explicaciones.

In [ ]:
# Identificamos los días extremos de ambas series

for columna, descripcion in [('con_lesionados', 'incidentes con lesionados'), ('con_fallecidos', 'incidentes fatales')]:
    print(f'\n{"=" * 55}')
    print(f'Serie: {descripcion}')
    print(f'{"=" * 55}')

    print('\n10 días con más incidentes:')
    print(series_diarias.nlargest(10, columna)[['con_lesionados', 'con_fallecidos']])

    print('\n10 días con menos incidentes:')
    print(series_diarias.nsmallest(10, columna)[['con_lesionados', 'con_fallecidos']])

El **2 de febrero**, Día de la Candelaria, aparece entre los máximos de incidentes con lesionados. ¿Fue una fecha realmente atípica o simplemente un viernes con muchos registros?

In [ ]:
# Agregamos el día de la semana a las series
series_comparacion = series_diarias.copy()

series_comparacion['dia_semana'] = series_comparacion.index.day_name()

# Promedio de incidentes con lesionados por día de la semana
promedios_dia = series_comparacion.groupby('dia_semana')['con_lesionados'].mean()

# Fechas que queremos investigar
fechas_interes = [
    '2024-02-02',
    '2024-02-03',
    '2024-02-10',
    '2024-04-19',
    '2024-09-16',
    '2024-12-25'
]

comparacion_fechas = series_comparacion.loc[fechas_interes].copy()

comparacion_fechas['promedio_mismo_dia'] = comparacion_fechas['dia_semana'].map(promedios_dia)

comparacion_fechas['diferencia_pct'] = ((comparacion_fechas['con_lesionados'] / comparacion_fechas['promedio_mismo_dia']) - 1) * 100

with pd.option_context('display.max_columns', None, 'display.width', 1000):
    print(comparacion_fechas.round(2))

El **19 de abril** destaca aún más ya que registró **50.24%** más incidentes con lesionados que el viernes promedio. En cambio, el **16 de septiembre** y el **25 de diciembre** estuvieron por debajo de los promedios de sus respectivos días. Revisemos en qué bloques horarios aparecieron esas diferencias.

In [ ]:
# Zoom en las horas para ver si es coincidente con descubrimientos anteriores

# Filtramos las fechas seleccionadas
df_fechas_interes = df_temporal[df_temporal['fecha_evento'].isin(pd.to_datetime(fechas_interes))].copy()

# Número de incidentes por fecha y bloque horario
tabla_fechas_horarios = pd.crosstab(
    df_fechas_interes['fecha_evento'], df_fechas_interes['bloque_horario']).reindex(
    index=pd.to_datetime(fechas_interes),
    columns=['Madrugada', 'Mañana', 'Tarde', 'Noche'],
    fill_value=0
)

# Porcentaje de incidentes correspondiente a cada bloque
tabla_fechas_pct = tabla_fechas_horarios.div(tabla_fechas_horarios.sum(axis=1), axis=0) * 100

print(tabla_fechas_horarios,"\n")
print(tabla_fechas_pct.round(2))

In [ ]:
# Utilizamos únicamente incidentes con lesionados
df_lesionados_temporal = df_temporal[df_temporal['con_lesionados']].copy()

orden_bloques = ['Madrugada', 'Mañana', 'Tarde', 'Noche']

# Promedio por bloque horario y día de la semana,
# considerando todas las fechas del calendario
conteos_diarios = (
    df_lesionados_temporal.groupby(['fecha_evento', 'bloque_horario']).size().unstack(fill_value=0)
    .reindex(index=calendario_2024, columns=orden_bloques, fill_value=0)
)

conteos_diarios['dia_semana'] = conteos_diarios.index.dayofweek

promedios_habituales = conteos_diarios.groupby('dia_semana')[orden_bloques].mean()

# Comparamos las fechas seleccionadas con su día habitual
comparacion_horaria = []

for fecha in pd.to_datetime(fechas_interes):
    dia = fecha.dayofweek

    for bloque in orden_bloques:
        observado = conteos_diarios.loc[fecha, bloque]
        habitual = promedios_habituales.loc[dia, bloque]

        comparacion_horaria.append({
            'fecha': fecha.date(),
            'bloque': bloque,
            'observado': observado,
            'promedio_habitual': habitual,
            'diferencia_pct': (observado / habitual - 1) * 100
        })

comparacion_horaria = pd.DataFrame(comparacion_horaria)

print(comparacion_horaria.round(2))

### ¿A qué hora ocurrieron los picos?

Al comparar seis fechas con el promedio del mismo día de la semana y bloque horario, encontramos comportamientos distintos. El **19 de abril**, máximo anual, registró **55 incidentes con lesionados durante la noche**, frente a **26.46** habituales para un viernes (**+107.85%**).

El **2 de febrero** destacó sobre todo en la mañana (**44 incidentes, +72.03%**). En cambio, el **16 de septiembre** y **Navidad** registraron menos incidentes diurnos que lo habitual, aunque sus madrugadas estuvieron por encima del promedio.

Así, un día con pocos incidentes en total **puede presentar un aumento en un bloque concreto** mostrando que no todos los picos diarios tienen el mismo perfil horario.

In [ ]:
# Periodos escolares y fechas investigadas

import matplotlib.dates as mdates

# Periodos vacacionales y colores
periodos_vacacionales = {
    'Semana Santa': {
        'inicio': '2024-03-25',
        'fin': '2024-04-05',
        'color': '#98B68E'    # Verde claro
    },
    'Verano': {
        'inicio': '2024-07-17',
        'fin': '2024-08-25',
        'color': '#F4A582'    # Amarillo claro
    },
    'Invierno': {
        'inicio': '2024-12-19',
        'fin': '2024-12-31',
        'color': '#9ECAE1'    # Azul claro
    }
}

fechas_destacadas = {
    '2024-02-02': {
        'etiqueta': 'Candelaria',
        'desplazamiento': (0, 15)
    },
    '2024-04-19': {
        'etiqueta': 'Máximo anual',
        'desplazamiento': (20, -24)
    },
    '2024-09-16': {
        'etiqueta': 'Independencia',
        'desplazamiento': (0, -22)
    },
    '2024-12-25': {
        'etiqueta': 'Navidad',
        'desplazamiento': (-28, 18)
    }
}

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    series_diarias.index,
    series_diarias['con_lesionados'],
    color='#397C8C',
    linewidth=1.5,
    label='Incidentes diarios',
    zorder=2
)

ax.axhline(
    series_diarias['con_lesionados'].mean(),
    color='#C56C54',
    linestyle='--',
    linewidth=1.4,
    label='Promedio diario',
    zorder=2
)

# Franjas vacacionales
for nombre, periodo in periodos_vacacionales.items():
    ax.axvspan(
        pd.Timestamp(periodo['inicio']),
        pd.Timestamp(periodo['fin']) + pd.Timedelta(days=1),
        color=periodo['color'],
        alpha=0.23,
        label=nombre,
        zorder=0
    )

# Fechas destacadas
for fecha_str, datos in fechas_destacadas.items():

    fecha = pd.Timestamp(fecha_str)
    valor = series_diarias.loc[fecha, 'con_lesionados']

    ax.scatter(
        fecha,
        valor,
        color='#C56C54',
        s=45,
        edgecolor='white',
        linewidth=0.8,
        zorder=5
    )

    ax.annotate(
        datos['etiqueta'],
        xy=(fecha, valor),
        xytext=datos['desplazamiento'],
        textcoords='offset points',
        ha='center',
        va='center',
        fontsize=9,
        arrowprops={
            'arrowstyle': '-',
            'color': '#777777',
            'lw': 0.8
        }
    )

ax.set_title(
    'Incidentes con lesionados y periodos vacacionales - CDMX 2024',
    pad=14
)

ax.set_xlabel('Fecha')
ax.set_ylabel('Incidentes con lesionados')

ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))

ax.set_xlim(
    pd.Timestamp('2024-01-01'),
    pd.Timestamp('2025-01-01')
)

ax.set_ylim(0, 145)

ax.grid(axis='y', alpha=0.2)
ax.set_axisbelow(True)

# Leyenda fuera del área de datos
ax.legend(
    loc='upper center',
    bbox_to_anchor=(0.5, -0.15),
    ncol=5,
    frameon=False,
    fontsize=9
)

plt.tight_layout()
plt.show()

### Fechas señaladas y vacaciones escolares

Superponemos los periodos de receso escolare sobre la serie diaria para explorar si coinciden con cambios más prolongados. Visualmente, **semana santa en Abril, Julio, Agosto y el periodo de Navidad** muestran una caída en los incidentes con lesionados ¿quizá podría estar relacionado con la disminución de vehículos que transitan en la ciudad?.

Marcamos además las fechas que acabamos de investigar: Candelaria, el máximo anual del **19 de abril**, Independencia y Navidad. La coincidencia con una celebración o un receso no basta para explicar el cambio asi que comprobaremos el contraste numéricamente.

In [ ]:
df_vacaciones = series_diarias[['con_lesionados']].copy()

df_vacaciones['dia_semana'] = df_vacaciones.index.dayofweek
df_vacaciones['periodo'] = 'Fuera de vacaciones'

# Asignamos cada fecha a su periodo correspondiente
for nombre, datos in periodos_vacacionales.items():
    inicio = pd.Timestamp(datos['inicio'])
    fin = pd.Timestamp(datos['fin'])

    df_vacaciones.loc[inicio:fin, 'periodo'] = nombre

# Promedio habitual por día de la semana, fuera de vacaciones
promedio_referencia = df_vacaciones[df_vacaciones['periodo'] == 'Fuera de vacaciones'].groupby('dia_semana')['con_lesionados'].mean()

# Referencia ajustada al día de la semana de cada fecha
df_vacaciones['esperado'] = df_vacaciones['dia_semana'].map(promedio_referencia)

# Comparación de periodos
comparacion_vacaciones = (
    df_vacaciones[df_vacaciones['periodo'] != 'Fuera de vacaciones'].groupby('periodo')
    .agg(
        dias=('con_lesionados', 'size'),
        promedio_observado=('con_lesionados', 'mean'),
        promedio_esperado=('esperado', 'mean')
    )
)

comparacion_vacaciones['diferencia_pct'] = (comparacion_vacaciones['promedio_observado'] / comparacion_vacaciones['promedio_esperado'] - 1) * 100

print(comparacion_vacaciones.round(2))

### ¿Bajan los registros durante las vacaciones?

Al comparar el promedio diario con una referencia formada por días **fuera de vacaciones** y ajustada al día de la semana, los incidentes con lesionados resultaron menores en los tres periodos: **Semana Santa, −12.03%**; **verano, −11.17%**; e **invierno, −10.23%**.

El parecido entre los resultados respalda la disminución observada, aunque el ajuste no controla todos los cambios propios de cada mes. El periodo de invierno incluye solo los días de la temporada dentro de 2024.

In [ ]:
# Incorporamos los incidentes fatales a la comparación vacacional
df_vacaciones['con_fallecidos'] = series_diarias['con_fallecidos']

# Referencia fuera de vacaciones, ajustada por día de la semana
referencia_fatales = (df_vacaciones[df_vacaciones['periodo'] == 'Fuera de vacaciones']
    .groupby('dia_semana')['con_fallecidos'].mean())

df_vacaciones['fatales_esperados'] = df_vacaciones['dia_semana'].map(referencia_fatales)

# Comparación por periodo vacacional
comparacion_fatales_vacaciones = (
    df_vacaciones[df_vacaciones['periodo'] != 'Fuera de vacaciones'].groupby('periodo')
    .agg(
        dias=('con_fallecidos', 'size'),
        incidentes_fatales=('con_fallecidos', 'sum'),
        promedio_observado=('con_fallecidos', 'mean'),
        promedio_esperado=('fatales_esperados', 'mean')
    )
)

comparacion_fatales_vacaciones['diferencia_pct'] = (comparacion_fatales_vacaciones['promedio_observado'] / comparacion_fatales_vacaciones['promedio_esperado'] - 1) * 100

with pd.option_context('display.max_columns', None, 'display.width', 1000):
    print(comparacion_fatales_vacaciones.round(2))

### ¿Y los incidentes fatales durante las vacaciones?

Aquí el patrón no se repite: **Semana Santa +14.65%**, **invierno +29.31%** y **verano −10.96%** frente a sus referencias ajustadas.

En Semana Santa hubo **18 incidentes fatales** y en invierno **24**. Son conteos pequeños para extraer conclusiones firmes a partir de los porcentajes. Lo que sí observamos es que las dos series no se comportan necesariamente igual durante los recesos escolares.

## ¿Qué aportan `prioridad` y `origen`?

Antes de seguir con la infraestructura, revisaremos dos variables del registro: la prioridad asignada al incidente y el canal por el que llegó el reporte.

In [ ]:
for columna in ['origen', 'prioridad']:
    print(f'\n{"=" * 45}')
    print(f'Variable: {columna}')
    print(f'{"=" * 45}')

    resumen = pd.DataFrame({
        'incidentes': df_temporal[columna].value_counts(dropna=False),
        'porcentaje': df_temporal[columna].value_counts(dropna=False, normalize=True) * 100
    })

    print(resumen.round(2))

In [ ]:
# Relación de prioridad con fallecimientos
tabla_prioridad = pd.crosstab(df_temporal['prioridad'], df_temporal['con_fallecidos'], margins=True)

tabla_prioridad.columns = [
    'Sin fallecimientos',
    'Con fallecimientos',
    'Total'
]

print(tabla_prioridad)

### Prioridad registrada

Los **511 casos de prioridad alta** registraron al menos un fallecimiento; solo **un incidente fatal** aparece con prioridad media. Ya que no conocemos los criterios operativos con los que se asignan los criterios de `prioridad`, no podemos afirmar cómo ni cuándo se asignó esa clasificación.

Entonces no profundizaremos en ella. Para un futuro modelo de consecuencias fatales, habría que revisar su posible **fuga de información (*data leakage*)**.

In [ ]:
# Agrupar canales de reporte
canales_principales = [
    'LLAMADA DEL 911',
    'RADIO',
    'BOTÓN DE AUXILIO'
]

df_temporal['origen_grupo'] = df_temporal['origen'].where(df_temporal['origen'].isin(canales_principales), 'Otros canales')

orden_bloques = ['Madrugada', 'Mañana', 'Tarde', 'Noche']

# Distribución absoluta por bloque horario
tabla_origen_horario = pd.crosstab(df_temporal['bloque_horario'], df_temporal['origen_grupo']).reindex(orden_bloques)

tabla_origen_pct = tabla_origen_horario.div(tabla_origen_horario.sum(axis=1), axis=0) * 100

print(tabla_origen_horario, "\n")
print(tabla_origen_pct.round(2))

### Origen del reporte

El **81.90%** de los incidentes llegó mediante llamadas al 911, seguido de radio (**11.19%**) y botón de auxilio (**5.63%**).

La composición cambia un poco con el horario: radio representa **13.45%** de los reportes en la madrugada y **9.67%** en la noche; el 911 pasa de **78.85%** a **83.46%**. Como las diferencias son moderadas y describen el canal de reporte, no crearemos otra gráfica y continuaremos nuestra exploración.

## ¿Qué relación tienen las vialidades con las consecuencias?

Ahora queremos comparar tipos de vialidad, geometrías de intersección y semaforización. Conocer la frecuencia no será suficiente: también observaremos personas lesionadas por 100 incidentes y la proporción que registra al menos un fallecimiento.

In [ ]:
columnas_vialidad = [
    'clasificacion_de_la_vialidad',
    'tipo_de_interseccion',
    'interseccion_semaforizada'
]

for columna in columnas_vialidad:
    print(f'\n{"=" * 55}')
    print(f'Variable: {columna}')
    print(f'{"=" * 55}')

    resumen = pd.DataFrame({
        'incidentes': df_temporal[columna].value_counts(dropna=False),
        'porcentaje': df_temporal[columna].value_counts(normalize=True, dropna=False) * 100
    })

    print(resumen.round(2))

Podemos ver que las **vías secundarias** concentran más registros y también que hay más incidentes en ubicaciones clasificadas como no semaforizadas. Eso no nos dice da información sobre dónde ocurren las consecuencias más graves. Asi que exploraremos la clasificación vial.

In [ ]:
resumen_vialidades = (
    df_temporal.groupby('clasificacion_de_la_vialidad')
    .agg(
        incidentes=('folio', 'count'),
        personas_lesionadas=('personas_lesionadas', 'sum'),
        personas_fallecidas=('personas_fallecidas', 'sum'),
        incidentes_fatales=('con_fallecidos', 'sum')
    )
)

# Indicadores normalizados por número de incidentes
resumen_vialidades['lesionados_por_100'] = resumen_vialidades['personas_lesionadas'] / resumen_vialidades['incidentes'] * 100

resumen_vialidades['pct_fatales'] = resumen_vialidades['incidentes_fatales'] / resumen_vialidades['incidentes'] * 100

resumen_vialidades['fallecidos_por_100'] = resumen_vialidades['personas_fallecidas'] / resumen_vialidades['incidentes'] * 100

resumen_vialidades = resumen_vialidades.sort_values('incidentes', ascending=False)

with pd.option_context('display.max_columns', None, 'display.width', 1000):
    print(resumen_vialidades.round(2))

Los **accesos carreteros** registran la mayor proporción fatal (**5.10%**) y **130.23 personas lesionadas por 100 incidentes**, pese a no estar entre las vialidades con más registros. Crearemos un gráfico para comparar visualmente.

In [ ]:
# Mismo orden de vialidades para ambas gráficas
datos_vialidades = resumen_vialidades.sort_values('incidentes',ascending=True)

fig, axes = plt.subplots(
    1,2,figsize=(12,5),sharey=True,
    gridspec_kw={'width_ratios':[1,1.1]}
)

# Incidentes registrados
axes[0].barh(
    datos_vialidades.index,datos_vialidades['incidentes'],
    color='#6E8898',height=0.6
)

for i, valor in enumerate(datos_vialidades['incidentes']):
    axes[0].text(valor+80,i,f'{valor:,}',va='center',fontsize=9)

axes[0].set_title('Incidentes registrados')
axes[0].set_xlabel('Número de incidentes')
axes[0].set_ylabel('Clasificación de la vialidad')
axes[0].set_xlim(0,datos_vialidades['incidentes'].max()*1.18)

# Proporción de incidentes fatales
axes[1].hlines(
    y=datos_vialidades.index,
    xmin=0,xmax=datos_vialidades['pct_fatales'],
    color='#C56C54',linewidth=2
)

axes[1].scatter(
    datos_vialidades['pct_fatales'],datos_vialidades.index,
    color='#C56C54',s=65,zorder=3
)

for i, valor in enumerate(datos_vialidades['pct_fatales']):
    axes[1].text(valor+0.12,i,f'{valor:.2f}%',va='center',fontsize=9)

axes[1].set_title('Incidentes con fallecimientos')
axes[1].set_xlabel('Incidentes fatales (%)')
axes[1].set_xlim(0,datos_vialidades['pct_fatales'].max()*1.18)

# Formato compartido
for ax in axes:
    ax.grid(axis='x',alpha=0.2)
    ax.set_axisbelow(True)

fig.suptitle(
    'Frecuencia y proporción fatal por clasificación de vialidad - CDMX 2024',
    fontsize=13
)

plt.tight_layout()
plt.show()

### Volumen y proporción fatal según la vialidad

Las **vías secundarias** concentran más incidentes (**9,761**), seguidas de los **ejes viales** (**9,184**). Pero presentan la menor proporción fatal (**0.86%**).

En el extremo contrario, los **accesos carreteros** registran **5.10%** de incidentes fatales, seguidos por las vías de acceso controlado radiales (**3.45%**) y anulares (**2.74%**). Este contraste nos muestra por qué conviene observar frecuencia y consecuencias por separado peor debemos recordar que estas proporciones se calculan **entre incidentes registrados** y no entre todos los viajes realizados en esas vías.

In [ ]:
resumen_intersecciones = (
    df_temporal.groupby('tipo_de_interseccion')
    .agg(
        incidentes=('folio', 'count'),
        personas_lesionadas=('personas_lesionadas', 'sum'),
        personas_fallecidas=('personas_fallecidas', 'sum'),
        incidentes_fatales=('con_fallecidos', 'sum')
    )
)

resumen_intersecciones['lesionados_por_100'] = resumen_intersecciones['personas_lesionadas'] / resumen_intersecciones['incidentes'] * 100
resumen_intersecciones['pct_fatales'] = resumen_intersecciones['incidentes_fatales'] / resumen_intersecciones['incidentes'] * 100
resumen_intersecciones = resumen_intersecciones.sort_values('incidentes', ascending=False)

with pd.option_context('display.max_columns', None, 'display.width', 1000):
    print(resumen_intersecciones.round(2))

### ¿Importa la forma del tramo o intersección?

Ahora repetiremos la comparación para las categorías `CRUZ`, `T`, `RECTA`, `CURVA`, `GLORIETA` y las demás configuraciones.

In [ ]:
datos_intersecciones = resumen_intersecciones.sort_values('incidentes',ascending=True)

fig,axes = plt.subplots(
    1,2,figsize=(12,5.5),sharey=True,
    gridspec_kw={'width_ratios':[1,1.1]}
)

# Incidentes registrados
axes[0].barh(
    datos_intersecciones.index,datos_intersecciones['incidentes'],
    color='#7184AE',height=0.65
)

for i,valor in enumerate(datos_intersecciones['incidentes']):
    axes[0].text(valor+120,i,f'{valor:,}',va='center',fontsize=9)

axes[0].set_title('Incidentes registrados')
axes[0].set_xlabel('Número de incidentes')
axes[0].set_ylabel('Tipo de intersección')
axes[0].set_xlim(0,datos_intersecciones['incidentes'].max()*1.15)

# Proporción de incidentes fatales
axes[1].hlines(
    y=datos_intersecciones.index,
    xmin=0,xmax=datos_intersecciones['pct_fatales'],
    color='#C56C54',linewidth=2
)

axes[1].scatter(
    datos_intersecciones['pct_fatales'],datos_intersecciones.index,
    color='#C56C54',s=65,zorder=3
)

for i,valor in enumerate(datos_intersecciones['pct_fatales']):
    axes[1].text(valor+0.12,i,f'{valor:.2f}%',va='center',fontsize=9)

axes[1].set_title('Incidentes con fallecimientos')
axes[1].set_xlabel('Incidentes fatales (%)')
axes[1].set_xlim(0,datos_intersecciones['pct_fatales'].max()*1.18)

# Formato compartido
for ax in axes:
    ax.grid(axis='x',alpha=0.2)
    ax.set_axisbelow(True)

fig.suptitle(
    'Frecuencia y proporción fatal por tipo de intersección - CDMX 2024',
    fontsize=13
)

plt.tight_layout()
plt.show()

### Consecuencias según el tipo de intersección

Las categorías **CRUZ** (**12,464**), **T** (**7,540**) y **RECTA** (**6,816**) concentran más incidentes. En proporción fatal destacan **GAZA (5.00%)**, **DESNIVEL (4.55%)** y **CURVA (4.15%)**; CRUZ presenta **0.91%**.

Los porcentajes de GAZA y DESNIVEL corresponden a solo **4 de 80** y **6 de 132** incidentes, respectivamente. En cambio, CURVA registra **41 de 988** y RECTA **195 de 6,816**.

Una explicación posible podría involucrar la velocidad de circulación y el diseño vial de las vías que presentan cada tipo de interseccion, pero nuestros datos no cuentan con registros de este estilo para corroborarlo.

### ¿Qué cambia cuando hay semáforo?

Primero compararemos los incidentes clasificados con y sin semáforo; después veremos qué ocurre **dentro de cada tipo de intersección**.

In [ ]:
resumen_semaforos = (
    df_temporal.groupby('interseccion_semaforizada').agg(
        incidentes=('folio', 'count'),
        personas_lesionadas=('personas_lesionadas', 'sum'),
        personas_fallecidas=('personas_fallecidas', 'sum'),
        incidentes_fatales=('con_fallecidos', 'sum')
    )
)

resumen_semaforos['lesionados_por_100'] = resumen_semaforos['personas_lesionadas'] / resumen_semaforos['incidentes'] * 100

resumen_semaforos['pct_fatales'] = resumen_semaforos['incidentes_fatales'] / resumen_semaforos['incidentes'] * 100

with pd.option_context('display.max_columns', None, 'display.width', 1000):
    print(resumen_semaforos.round(2))

Sin semáforo encontramos **397 incidentes fatales entre 20,737 (1.91%)**, por el contrario, con semáforo, **115 entre 9,918 (1.16%)**. En cambio, los lesionados por 100 incidentes son ligeramente mayores en las ubicaciones semaforizadas.

Pero una recta sin semáforo y un cruce semaforizado pueden tener condiciones muy diferentes. Para afinar la comparación, separaremos las geometrías de los cruces.

In [ ]:
# Resumen por geometría y presencia de semáforo
resumen_cruce_semaforos = (df_temporal.groupby(['tipo_de_interseccion', 'interseccion_semaforizada'])
    .agg(
        incidentes=('folio', 'count'),
        incidentes_fatales=('con_fallecidos', 'sum'),
        personas_lesionadas=('personas_lesionadas', 'sum')
    )
)

# Porcentaje de incidentes con fallecimientos
resumen_cruce_semaforos['pct_fatales'] = resumen_cruce_semaforos['incidentes_fatales'] / resumen_cruce_semaforos['incidentes'] * 100

# Personas lesionadas por 100 incidentes
resumen_cruce_semaforos['lesionados_por_100'] = resumen_cruce_semaforos['personas_lesionadas'] / resumen_cruce_semaforos['incidentes'] * 100

display(resumen_cruce_semaforos.round(2))

### Semáforo y geometría: la diferencia no siempre va en la misma dirección

En `RECTA`, la proporción fatal es **3.00% sin semáforo** (186 de 6,190) y **1.44% con semáforo** (9 de 626). En `CRUZ` ocurre una pequeña inversión: **0.77%** sin semáforo frente a **1.02%** con él.

Esto nos deja ver que la diferencia global entre lugares semaforizados y no semaforizados depende, al menos en parte, de **qué configuraciones de cruces estamos comparando**. También hay categorías con pocos casos, como los 18 registros semaforizados en DESNIVEL que evitaremos destacarlas por su porcentaje aislado.

In [ ]:
# Comparación de incidentes fatales dentro de cada tipo de intersección
datos_semaforos = resumen_cruce_semaforos.reset_index()

# Seleccionamos geometrías con al menos 100 registros
# tanto con semáforo como sin semáforo
conteos = datos_semaforos.pivot(index='tipo_de_interseccion', columns='interseccion_semaforizada', values='incidentes')

categorias_validas = conteos[(conteos['SI'] >= 100) & (conteos['NO'] >= 100)].index

datos_plot = datos_semaforos[datos_semaforos['tipo_de_interseccion'].isin(categorias_validas)].copy()

orden = (datos_plot[datos_plot['interseccion_semaforizada'] == 'NO'].sort_values('pct_fatales', ascending=False)['tipo_de_interseccion'].tolist())

fig, ax = plt.subplots(figsize=(8, 5))

sns.barplot(
    data=datos_plot,
    y='tipo_de_interseccion',
    x='pct_fatales',
    hue='interseccion_semaforizada',
    order=orden,
    hue_order=['NO', 'SI'],
    palette={'NO': '#C56C54', 'SI': '#397C8C'},
    orient='h',
    ax=ax
)

for contenedor in ax.containers:
    ax.bar_label(
        contenedor,
        fmt='%.2f%%',
        padding=3,
        fontsize=9
    )

ax.set_title('Incidentes fatales por geometría y semaforización - CDMX 2024')
ax.set_xlabel('Incidentes con fallecimientos (%)')
ax.set_ylabel('Tipo de intersección')
ax.set_xlim(0, 5.3)
ax.grid(axis='x', alpha=0.2)
ax.set_axisbelow(True)
ax.legend(title='Semáforo', loc='upper left', bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()

### ¿Los vehículos aparecen en las mismas vialidades?

Ya encontramos diferencias entre tipos de vehículo y entre tipos de vialidad. Ahora cruzaremos ambas variables para ver si ciertas combinaciones destacan.

In [ ]:
# Asociamos cada incidente con su grupo de vehículo y la clasificación de la vialidad
df_vehiculos_vialidad = df_vehiculos[['folio', 'tipo_vehiculo_grupo']].merge(dfs['hechos_ssc'][['folio', 'clasificacion_de_la_vialidad']],
    on='folio', how='inner', validate='one_to_one')

# Verificamos que no se hayan perdido incidentes
print('Registros:', len(df_vehiculos_vialidad))
print('Folios únicos:', df_vehiculos_vialidad['folio'].nunique())

# Frecuencia de grupos de vehículo según clasificación vial
tabla_vehiculos_vialidad = pd.crosstab(df_vehiculos_vialidad['tipo_vehiculo_grupo'], df_vehiculos_vialidad['clasificacion_de_la_vialidad'])

tabla_vehiculos_vialidad_pct = tabla_vehiculos_vialidad.div(tabla_vehiculos_vialidad.sum(axis=1), axis=0) * 100

with pd.option_context('display.max_columns', None, 'display.width', 1000):
  print(tabla_vehiculos_vialidad)
  print('\nDistribución porcentual por grupo de vehículo:')
  print(tabla_vehiculos_vialidad_pct.round(2))

En los incidentes asociados con vehículos de **carga**, los accesos carreteros representan **4.30%**, frente a **2.56%** en el conjunto general. Es una diferencia en la distribución de **ese grupo** pero todavía no nos dice qué porcentaje de todos los incidentes carreteros corresponde a carga.

In [ ]:
# Excluimos los vehículos sin clasificación
datos_heatmap = tabla_vehiculos_vialidad_pct.drop(index='Sin dato')

# Orden editorial de las vialidades
orden_vialidades = [
    'VIA SECUNDARIA',
    'EJE VIAL',
    'VIA PRIMARIA',
    'VAC ANULAR',
    'VAC RADIAL',
    'ACCESO CARRETERO',
    'VAC VIADUCTO'
]

datos_heatmap = datos_heatmap[orden_vialidades]

fig, ax = plt.subplots(figsize=(10, 4.5))

sns.heatmap(
    datos_heatmap,
    annot=True,
    fmt='.1f',
    cmap='YlOrRd',
    linewidths=0.5,
    cbar_kws={
        'label': 'Distribución de incidentes (%)'
    },
    ax=ax
)

ax.set_title('Incidentes por intersecciones viales a lo largo de categorías vehiculares - CDMX 2024')
ax.set_xlabel('Clasificación de la vialidad')
ax.set_ylabel('Grupo de vehículo')
ax.tick_params(axis='x', rotation=35)
ax.tick_params(axis='y', rotation=0)
plt.tight_layout()
plt.show()

Ahora invertiremos el denominador para saber de todos los incidentes registrados en **cada clasificación vial**, ¿qué proporción corresponde a motocicletas, transporte de personas, carga y otros grupos?

In [ ]:
tabla_vialidad_vehiculos_pct = tabla_vehiculos_vialidad.div(tabla_vehiculos_vialidad.sum(axis=0), axis=1) * 100

# Excluimos "Sin dato" solo de la visualización,
datos_heatmap = tabla_vialidad_vehiculos_pct.drop(index='Sin dato')
datos_heatmap = datos_heatmap[orden_vialidades]

fig, ax = plt.subplots(figsize=(10, 4.5))

sns.heatmap(
    datos_heatmap,
    annot=True,
    fmt='.1f',
    cmap='YlOrRd',
    linewidths=0.5,
    cbar_kws={'label': 'Participación en incidentes de la vialidad (%)'},
    ax=ax
)

ax.set_title('Composición vehicular de los incidentes por vialidad - CDMX 2024')
ax.set_xlabel('Clasificación de la vialidad')
ax.set_ylabel('Grupo de vehículo')
ax.tick_params(axis='x', rotation=35)
ax.tick_params(axis='y', rotation=0)
plt.tight_layout()
plt.show()

### Composición vehicular según la vialidad

Las **motocicletas** representan más de la mitad de los incidentes en las siete clasificaciones viales. La micromovilidad tiene más presencia relativa en vías secundarias (**9.1%**) y menos en accesos carreteros (**2.2%**).

En los accesos carreteros, carga representa solo **2.8%** de los registros. Su mayor proporción fatal no parece explicarse por una presencia masiva de ese grupo. Para comprobarlo, revisaremos los incidentes fatales de esos accesos según el vehículo registrado.

### Una mirada a los accesos carreteros

In [ ]:
# Incorporamos las consecuencias al cruce vehículo-vialidad
df_vehiculos_vialidad_fatal = df_vehiculos_vialidad.merge(
    dfs['hechos_ssc'][['folio', 'personas_fallecidas', 'personas_lesionadas']],
    on='folio', validate='one_to_one')

# Seleccionamos accesos carreteros
accesos = df_vehiculos_vialidad_fatal.loc[
    df_vehiculos_vialidad_fatal['clasificacion_de_la_vialidad'] == 'ACCESO CARRETERO']

resumen_accesos = (
    accesos.groupby('tipo_vehiculo_grupo')
    .agg(
        incidentes=('folio', 'count'),
        incidentes_fatales=('personas_fallecidas', lambda x: (x > 0).sum()),
        personas_fallecidas=('personas_fallecidas', 'sum')
    )
)

resumen_accesos['pct_fatales'] = resumen_accesos['incidentes_fatales'] / resumen_accesos['incidentes'] * 100

with pd.option_context('display.max_columns', None, 'display.width', 1000):
    print(resumen_accesos.sort_values('incidentes_fatales', ascending=False).round(2))

### ¿Qué vehículos aparecen en los incidentes fatales de accesos carreteros?

De los **40 incidentes fatales** registrados en accesos carreteros, **19** corresponden al grupo motocicleta y **13** a transporte de personas: juntos suman **80%**. Dentro de cada grupo, las proporciones fatales son similares (**4.52%** y **4.80%**).

Carga presenta **2 casos entre 22 registros (9.09%)**, y otros **5 incidentes fatales** pertenecen a `Sin dato`, por lo que no podemos atribuir la proporción fatal de los accesos carreteros a un único grupo.

Como en todo el análisis de vehículos, la categoría corresponde solo al **Vehículo 1** registrado lo cual limita la exploración al no tener datos de más vehículos involucrados.

## Exploración geográfica

Después de explorar cuándo ocurren los incidentes, qué vehículos aparecen y cómo cambian sus consecuencias según la vialidad, nos falta una pregunta: **¿dónde ocurren y cómo se concentran?**

Empezaremos con las ubicaciones registradas, distinguiendo las consecuencias de cada incidente. Después pasaremos a las superficies de densidad y a las proporciones territoriales para reconocer patrones que no se distinguen fácilmente entre miles de puntos. Cada mapa responderá una pregunta distinta.

In [ ]:
# Columnas geográficas
lista_mask = ['latitud', 'longitud', 'coord', 'alcaldia', 'colonia']
columnas_geo = [columna for columna in dfs['hechos_ssc'].columns if any(termino in columna.lower() for termino in lista_mask)]

print('Columnas geográficas disponibles:')
print(columnas_geo,'\n')

with pd.option_context('display.max_columns', None, 'display.width', 1000):
    print(dfs['hechos_ssc'][columnas_geo].head())

Comprobaremos que las **30,655 filas** siguen disponibles y observaremos cuántas ubicaciones comparten coordenadas.

In [ ]:
df_geo = dfs['hechos_ssc'].copy()

# Coordenadas disponibles
coordenadas_validas = (df_geo['latitud'].notna() & df_geo['longitud'].notna())

# Rango geográfico preliminar de CDMX
rango_cdmx = (df_geo['latitud'].between(19.0, 19.7) & df_geo['longitud'].between(-99.4, -98.8))

# Registros que cumplen ambas condiciones
df_geo['coordenadas_utilizables'] = (coordenadas_validas & rango_cdmx)

print('Total de incidentes:', len(df_geo))

print('Sin coordenadas completas:', (~coordenadas_validas).sum())

print('Fuera del rango preliminar:', (coordenadas_validas & ~rango_cdmx).sum())

print('Coordenadas utilizables:', df_geo['coordenadas_utilizables'].sum())

# Repetición de puntos entre registros con coordenadas utilizables
puntos_validos = df_geo.loc[df_geo['coordenadas_utilizables'], ['latitud', 'longitud']]

print('Coordenadas únicas:', len(puntos_validos.drop_duplicates()))

print('\nCoordenadas más repetidas:')
print(puntos_validos.value_counts().head(10))

### Mapa: Ubicaciones individuales y consecuencias registradas

Antes de resumir los incidentes mediante superficies de densidad, podemos explorar **dónde se registró cada uno**. En el mapa se agrupan los puntos cercanos cuando vemos la ciudad completa y se separan progresivamente al acercarnos, hasta permitir examinar las ubicaciones a nivel de calle.

Distinguimos tres categorías excluyentes: **fatal** cuando hubo al menos un fallecimiento, aunque también se registraran lesionados; **con lesionados** cuando hubo personas lesionadas pero ningún fallecimiento; y **sin víctimas registradas** cuando ambas cantidades fueron cero.

Los círculos numerados muestran **cuántos registros se agrupan visualmente a ese nivel de zoom**. Al hacer zoom aparecen los puntos individuales; al seleccionarlos podemos consultar las consecuencias registradas. Esto nos permitirá comparar las ubicaciones originales con las concentraciones que veremos a continuación.

In [ ]:
import folium
from folium.plugins import FastMarkerCluster, HeatMap

# Coordenadas validadas y variables compartidas por los mapas
df_mapa = df_geo.loc[df_geo['coordenadas_utilizables']].copy()
centro = [df_mapa['latitud'].median(),df_mapa['longitud'].median()]
ruta_mapas = ruta_proyecto/'content/maps/interactive'
ruta_mapas.mkdir(parents=True,exist_ok=True)

# Una categoría por incidente: fatal prevalece sobre lesionados
fatales = df_mapa['personas_fallecidas'].fillna(0)>0
lesionados = df_mapa['personas_lesionadas'].fillna(0)>0
n_fatales = int(fatales.sum())
n_lesionados = int((~fatales & lesionados).sum())
n_sin_victimas = int((~fatales & ~lesionados).sum())

puntos = df_mapa[
    ['latitud','longitud','personas_fallecidas','personas_lesionadas']
].fillna(0).values.tolist()

mapa_puntos = folium.Map(location=centro,zoom_start=11,tiles=None,control_scale=True)
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}',
    attr='Tiles © Esri',name='Esri World Street Map'
).add_to(mapa_puntos)

# Los grupos son neutros; solo los puntos individuales indican gravedad
FastMarkerCluster(
    puntos,name='Incidentes registrados',maxClusterRadius=55,
    disableClusteringAtZoom=17,
    icon_create_function="""
    function (cluster) {
        return L.divIcon({
            html: '<div style="background:#4D5866;color:white;border:2px solid white;'
                + 'border-radius:50%;height:38px;width:38px;display:flex;'
                + 'align-items:center;justify-content:center;font-size:12px;font-weight:600;'
                + 'box-shadow:0 1px 5px #5557;">' + cluster.getChildCount() + '</div>',
            className:'',iconSize:[42,42],iconAnchor:[21,21]
        });
    }
    """,
    callback="""
    function (row) {
        var fallecidos = Number(row[2]) || 0;
        var heridos = Number(row[3]) || 0;
        var color = fallecidos>0 ? '#C56C54' : heridos>0 ? '#2876B9' : '#808B96';
        var tipo = fallecidos>0 ? 'Fatal' : heridos>0 ? 'Con lesionados' : 'Sin víctimas registradas';
        var punto = L.circleMarker([row[0],row[1]],{
            radius:5,color:'#FFFFFF',weight:1,fillColor:color,fillOpacity:0.9
        });
        punto.bindPopup('<b>'+tipo+'</b><br>Personas lesionadas: '+heridos+
            '<br>Personas fallecidas: '+fallecidos);
        return punto;
    }
    """
).add_to(mapa_puntos)

# Leyenda para los puntos, distinta del conteo de los grupos
leyenda = f"""
<div style="position:fixed;bottom:28px;left:28px;z-index:9999;
    background:white;padding:10px 13px;border:1px solid #bbb;
    border-radius:6px;box-shadow:0 1px 6px #0003;font:12px Arial;">
    <b>Consecuencias registradas</b>
    <div style="margin-top:7px"><span style="color:#C56C54">●</span> Fatales: {n_fatales:,}</div>
    <div><span style="color:#2876B9">●</span> Con lesionados: {n_lesionados:,}</div>
    <div><span style="color:#808B96">●</span> Sin víctimas registradas: {n_sin_victimas:,}</div>
    <div style="color:#555;margin-top:6px">Círculos numerados = registros agrupados</div>
</div>
"""
mapa_puntos.get_root().html.add_child(folium.Element(leyenda))
folium.LayerControl().add_to(mapa_puntos)

ruta_html_puntos = ruta_mapas/'incident_locations_by_consequence_2024.html'
mapa_puntos.save(str(ruta_html_puntos))
display(mapa_puntos)
print(f'Mapa de puntos guardado en: {ruta_html_puntos}')

### ¿Qué significa densidad espacial suavizada?

Un punto en el mapa puede marcar la ubicación registrada de un incidente, sin embargo, cuando tenemos muchos puntos superpuestos es difícil reconocer las zonas de mayor concentración. Para observarlas utilizamos la **densidad espacial suavizada** que es una superficie que resume cómo se distribuyen los registros en el territorio, sin representar cada incidente como una mancha independiente.

- **¿Cómo la construimos?**. En los **mapas estáticos** construimos una cuadrícula de **250 × 250 m**, contamos los incidentes de cada celda y aplicamos un **filtro gaussiano con σ = 500 m**. El filtro reparte el peso de los conteos entre las celdas cercanas: las concentraciones aparecen más intensas y se atenúan gradualmente al alejarnos. Por último, dividimos el conteo suavizado de cada celda entre su área (**0.0625 km²**) para expresar la superficie en **incidentes/km²**.

- **¿Cómo interpretamos los colores?**. Una densidad de 60 incidentes/km² indica la intensidad *suavizada* estimada en esa posición; **no significa** que hayamos contado exactamente 60 incidentes dentro de un cuadrado de 1 km por lado. Tampoco implica que una mancha mida 1 km²: su extensión depende de la distribución de los puntos, del suavizado y del umbral utilizado para hacerla visible. Estos mapas nos permiten visualizar las concentraciones de incidentes.

No todos los mapas utilizan la misma escala visual. En los mapas metropolitanos podemos comparar patrones espaciales utilizando una cuadrícula y un suavizado comunes, pero las escalas de color pueden variar según la cantidad de incidentes que hay en cada zona.

En el **atlas de las 16 alcaldías**, además, calculamos la superficie con los registros de cada alcaldía y mostramos únicamente las concentraciones superiores a su **percentil 75 local**, ajustando su escala de color por separado. Por eso **el aspecto y tamaño de las manchas de color puede variar con la cantidad y distribución de incidentes de cada territorio**, es decir, un naranja intenso en una alcaldía no equivale necesariamente a la misma densidad absoluta en otra.

Finalmente, el **mapa de calor interactivo de Folium** sirve para explorar las concentraciones al desplazarnos y hacer zoom pero su intensidad depende de parámetros visuales en píxeles (*radius* y *blur*), por lo que **no debe leerse como una escala de incidentes/km²**, si no que la unidad cuantitativa corresponde a las superficies gaussianas construidas.

### Mapa: Densidad suavizada incidentes

Un mapa de calor permite recorrer la ciudad y observar concentraciones generales de registros. Su intensidad depende del zoom y los parámetros de visualización.

In [ ]:
# Crear mapa sin proveedor predeterminado
mapa_incidentes = folium.Map(
    location=centro,
    zoom_start=11,
    tiles=None,
    control_scale=True
)

# Mapa base Esri
folium.TileLayer(
    tiles=(
        "https://server.arcgisonline.com/ArcGIS/rest/services/"
        "World_Street_Map/MapServer/tile/{z}/{y}/{x}"
    ),
    attr="Tiles © Esri",
    name="Esri World Street Map"
).add_to(mapa_incidentes)

HeatMap(
    data=df_mapa[['latitud', 'longitud']].values.tolist(),
    radius=11,
    blur=15,
    min_opacity=0.25,
    max_zoom=15,
    name='Densidad de incidentes'
).add_to(mapa_incidentes)

folium.LayerControl().add_to(mapa_incidentes)

display(mapa_incidentes)

In [ ]:
ruta_html = ruta_mapas / 'incident_density_2024.html'
mapa_incidentes.save(str(ruta_html))

print(f'Mapa guardado en:\n{ruta_html}')
print(f'Tamaño: {ruta_html.stat().st_size / 1024 / 1024:.2f} MB')

### Del mapa interactivo al estático

Guardamos el primer mapa como **HTML** para poder ver las concentraciones de incidentes de forma interactiva. Ahora queremos una versión estática que también permita reconocer las calles y las zonas de la ciudad. Antes de crearla revisaremos los archivos de colonias que utilizamos durante la limpieza.

In [ ]:
# Extensiones de archivos geográficos
extensiones_geo = {
    '.geojson', '.json', '.shp',
    '.gpkg', '.zip', '.kml'
}

archivos_geo = [
    archivo
    for archivo in ruta_proyecto.rglob('*')
    if archivo.is_file()
    and archivo.suffix.lower() in extensiones_geo
]

print(f'Archivos geográficos encontrados: {len(archivos_geo)}\n')

for archivo in archivos_geo:
    print(archivo.relative_to(ruta_proyecto))

Los dos catálogos contienen polígonos de colonias, pero no tienen exactamente la misma cobertura. Usaremos el expandido (original) para dibujar los límites de las alcaldías, mientras que el reducido seguirá siendo nuestra referencia para los análisis por colonia.

In [ ]:
import geopandas as gpd

ruta_original = ruta_proyecto / 'data/raw/catalogo_colonias.json'
ruta_reducido = ruta_proyecto / 'data/raw/catalogo_colonias_reducido.json'

for nombre, ruta in [
    ('Catálogo original', ruta_original),
    ('Catálogo reducido', ruta_reducido)
]:
    print(f'\n{"=" * 55}')
    print(nombre)
    print(f'{"=" * 55}')

    gdf = gpd.read_file(ruta)

    print('Registros:', len(gdf))
    print('Columnas:', gdf.columns.tolist())
    print('CRS:', gdf.crs)

    print('\nTipos de geometría:')
    print(gdf.geometry.geom_type.value_counts())

    print('\nExtensión geográfica:')
    print(gdf.total_bounds)

    print('\nPrimeros registros:')
    print(gdf.head(3))

In [ ]:
# Cargamos ambos catálogos
gdf_original = gpd.read_file(ruta_proyecto / 'data/raw/catalogo_colonias.json')

gdf_reducido = gpd.read_file(ruta_proyecto / 'data/raw/catalogo_colonias_reducido.json')

# Convertimos nuestros incidentes en puntos geográficos
gdf_incidentes = gpd.GeoDataFrame(
    df_geo.copy(),
    geometry=gpd.points_from_xy(
        df_geo['longitud'],
        df_geo['latitud']
    ),
    crs='EPSG:4326'
)

# Unimos las geometrías de cada catálogo
territorio_original = gdf_original.geometry.union_all()
territorio_reducido = gdf_reducido.geometry.union_all()

# Comprobamos cuántos incidentes están dentro
dentro_original = gdf_incidentes.geometry.intersects(territorio_original)

dentro_reducido = gdf_incidentes.geometry.intersects(territorio_reducido)

print('Cobertura de los catálogos sobre los incidentes:\n')
print(f'Original: {dentro_original.sum():,} / {len(df_geo):,} 'f'({dentro_original.mean():.2%})')
print(f'Reducido: {dentro_reducido.sum():,} / {len(df_geo):,} 'f'({dentro_reducido.mean():.2%})')
print('\nIncidentes fuera de ambos catálogos:')
print((~dentro_original & ~dentro_reducido).sum())

La cobertura del catálogo original alcanza **30,491 de 30,655 incidentes (99.47%)** y la del reducido, **98.30%**. Como ninguno describe todo el territorio, no lo usaremos para recortar los puntos así que el mapa estático representará **los 30,655 registros** y añadirá los límites territoriales como contexto.

### Mapa estático de concentración

Para conservar las calles bajo la capa de densidad, utilizaremos un mapa base y un suavizado gaussiano de **500 m**. La escala mostrará densidad espacial suavizada en incidentes/km², no conteos directos por cuadrantes de 1 km².

El mapa base se obtiene de internet y se utiliza el servicio Esri para obtener el fondo, la atribución del proveedor se conserva en la figura.

In [ ]:
%pip -q install contextily

In [ ]:
import numpy as np
import contextily as cx
from scipy.ndimage import gaussian_filter
from matplotlib.colors import Normalize,LinearSegmentedColormap

ruta_salida = ruta_proyecto / 'content/maps/static'
ruta_salida.mkdir(parents=True,exist_ok=True)

crs_metrico = 'EPSG:32614'
gdf_puntos = gpd.GeoDataFrame(
    df_mapa[['folio','latitud','longitud']].copy(),
    geometry=gpd.points_from_xy(df_mapa['longitud'],df_mapa['latitud']),crs='EPSG:4326'
).to_crs(crs_metrico)
gdf_alcaldias = gdf_original.to_crs(crs_metrico).dissolve(by='NOMDT')

# Densidad espacial suavizada, expresada en incidentes por km²
resolucion_m = 250
suavizado_m = 500
limites = np.vstack([gdf_puntos.total_bounds,gdf_alcaldias.total_bounds])
xmin,ymin = limites[:,:2].min(axis=0) - 1500
xmax,ymax = limites[:,2:].max(axis=0) + 1500
bordes_x = np.arange(xmin,xmax+resolucion_m,resolucion_m)
bordes_y = np.arange(ymin,ymax+resolucion_m,resolucion_m)

conteos,_,_ = np.histogram2d(
    gdf_puntos.geometry.y,gdf_puntos.geometry.x,bins=[bordes_y,bordes_x]
)
densidad = gaussian_filter(conteos,sigma=suavizado_m/resolucion_m) / (resolucion_m/1000)**2

umbral = 1.5
valores = densidad[densidad >= umbral]
vmax = np.percentile(valores,99) if len(valores) else umbral+1
densidad_visible = np.ma.masked_less(densidad,umbral)
cmap = plt.colormaps['YlOrRd'].copy()
cmap.set_bad((0,0,0,0))

fig,ax = plt.subplots(figsize=(9,10),facecolor='white')
ax.set_xlim(bordes_x[0],bordes_x[-1])
ax.set_ylim(bordes_y[0],bordes_y[-1])

# Fondo cartográfico sin marcadores ni capas de puntos
proveedores = [
    ('Esri World Street Map',cx.providers.Esri.WorldStreetMap),
    ('OpenStreetMap',cx.providers.OpenStreetMap.Mapnik)
]
for nombre,proveedor in proveedores:
    try:
        cx.add_basemap(
            ax,source=proveedor,crs=crs_metrico,zoom=11,
            attribution_size=6,headers={'User-Agent':'CDMXRoadSafetyIntelligence/1.0 (educational map)'}
        )
        print('Mapa base:',nombre)
        break
    except Exception as error:
        print(f'No se pudo cargar {nombre}: {error}')
else:
    plt.close(fig)
    raise RuntimeError('No se pudo cargar ningún mapa base. Comprueba la conexión de Colab.')

img = ax.imshow(
    densidad_visible,extent=[bordes_x[0],bordes_x[-1],bordes_y[0],bordes_y[-1]],
    origin='lower',cmap=cmap,norm=Normalize(vmin=umbral,vmax=vmax),
    alpha=0.68,zorder=2
)
gdf_alcaldias.boundary.plot(ax=ax,color='#4D5866',linewidth=0.65,alpha=0.7,zorder=3)

ax.set_title('CIUDAD DE MÉXICO\nDensidad de incidentes viales · 2024',fontsize=15,pad=15)
ax.set_axis_off()

cbar = fig.colorbar(img,ax=ax,fraction=0.035,pad=0.02,extend='max')
cbar.set_label('Densidad espacial suavizada (incidentes/km²)',fontsize=9)
cbar.ax.tick_params(labelsize=8)

# Escala métrica en la proyección UTM
barra_x = bordes_x[0] + 0.07*(bordes_x[-1]-bordes_x[0])
barra_y = bordes_y[0] + 0.07*(bordes_y[-1]-bordes_y[0])
ax.plot([barra_x,barra_x+10000],[barra_y,barra_y],color='#30343B',linewidth=3,zorder=5)
ax.text(barra_x+5000,barra_y+700,'10 km',ha='center',fontsize=9,zorder=5,
        bbox={'facecolor':'white','edgecolor':'none','alpha':0.85,'pad':2})

fig.text(0.5,0.025, 'Fuente: SSC CDMX (2024) · Límites: catálogo de colonias',
         ha='center',fontsize=8,color='#555555')
fig.text(0.5,0.045, 'Densidad suavizada mediante filtro gaussiano (σ = 500 m)',
         ha='center',fontsize=8,color='#555555')

plt.tight_layout(rect=(0,0.045,1,1))

ruta_png = ruta_salida / 'mapa_densidad_incidentes_cdmx_2024.png'
ruta_svg = ruta_salida / 'mapa_densidad_incidentes_cdmx_2024.svg'
fig.savefig(ruta_png,dpi=300,bbox_inches='tight',facecolor='white')
fig.savefig(ruta_svg,bbox_inches='tight',facecolor='white')
plt.show()

print(f'Mapa guardado en:\n- {ruta_png}\n- {ruta_svg}')

En estos mapas podemos ver la densidad suavizada, así como los límites de las alcaldías procedentes del catálogo de colonias.

Con esta base podremos preparar mapas comparables para los incidentes fatales y los que involucran motocicletas, y después explorar las diferencias territoriales que encontramos en el EDA.

### Mapa: Distribución espacial de incidentes fatales

En los mapas generales podemos ver las zonas donde se concentran más registros. Ahora nos preguntamos si la distribución cambia al observar solamente los **512 incidentes con al menos un fallecimiento**.

Conservaremos el suavizado de **500 m** para mantener la misma escala espacial de análisis, aunque cada mapa tendrá su propia escala de color por la diferencia de magnitudes.

In [ ]:
# Incidentes con al menos una persona fallecida
folios_fatales = df_mapa.loc[
    df_mapa['personas_fallecidas'] > 0,'folio'
]

gdf_fatales = gdf_puntos[
    gdf_puntos['folio'].isin(folios_fatales)
].copy()

print(f'Incidentes fatales: {len(gdf_fatales):,}')

# Densidad espacial
conteos_fatales,_,_ = np.histogram2d(
    gdf_fatales.geometry.y,gdf_fatales.geometry.x,
    bins=[bordes_y,bordes_x]
)

densidad_fatal = gaussian_filter(conteos_fatales, sigma=suavizado_m/resolucion_m) / (resolucion_m/1000)**2

umbral_fatal = 0.02
valores = densidad_fatal[densidad_fatal >= umbral_fatal]
vmax_fatal = np.percentile(valores,99) if len(valores) else umbral_fatal+0.01

densidad_fatal_visible = np.ma.masked_less(densidad_fatal,umbral_fatal)

cmap_fatal = plt.colormaps['YlOrRd'].copy()
cmap_fatal.set_bad((0,0,0,0))

fig,ax = plt.subplots(figsize=(9,10),facecolor='white')
ax.set_xlim(bordes_x[0],bordes_x[-1])
ax.set_ylim(bordes_y[0],bordes_y[-1])

# Mapa base
proveedores = [
    ('Esri World Street Map',cx.providers.Esri.WorldStreetMap),
    ('OpenStreetMap',cx.providers.OpenStreetMap.Mapnik)
]

for nombre,proveedor in proveedores:
    try:
        cx.add_basemap(
            ax,source=proveedor,crs=crs_metrico,zoom=11,
            attribution_size=6
        )
        print('Mapa base:',nombre)
        break
    except Exception as error:
        print(f'No se pudo cargar {nombre}: {error}')
else:
    plt.close(fig)
    raise RuntimeError('No se pudo cargar ningún mapa base.')

# Densidad de incidentes fatales
img = ax.imshow(
    densidad_fatal_visible,
    extent=[bordes_x[0],bordes_x[-1],bordes_y[0],bordes_y[-1]],
    origin='lower',
    cmap=cmap_fatal,
    norm=Normalize(vmin=umbral_fatal,vmax=vmax_fatal),
    alpha=0.7,
    zorder=2
)

gdf_alcaldias.boundary.plot(
    ax=ax,color='#4D5866',
    linewidth=0.65,alpha=0.7,zorder=3
)

ax.set_title(
    'CIUDAD DE MÉXICO\nDensidad de incidentes fatales · 2024',
    fontsize=15,pad=15
)

ax.set_axis_off()

cbar = fig.colorbar(img,ax=ax,fraction=0.035,pad=0.02,extend='max')
cbar.set_label('Densidad espacial suavizada (incidentes/km²)',fontsize=9)
cbar.ax.tick_params(labelsize=8)

# Escala
barra_x = bordes_x[0] + 0.07*(bordes_x[-1]-bordes_x[0])
barra_y = bordes_y[0] + 0.07*(bordes_y[-1]-bordes_y[0])

ax.plot(
    [barra_x,barra_x+10000],[barra_y,barra_y],
    color='#30343B',linewidth=3,zorder=5
)

ax.text(
    barra_x+5000,barra_y+700,'10 km',
    ha='center',fontsize=9,zorder=5,
    bbox={'facecolor':'white','edgecolor':'none','alpha':0.85,'pad':2}
)

fig.text(0.5,0.025, 'Fuente: SSC CDMX (2024) · Límites: catálogo de colonias',
         ha='center',fontsize=8,color='#555555')
fig.text(0.5,0.045, 'Densidad suavizada mediante filtro gaussiano (σ = 500 m)',
         ha='center',fontsize=8,color='#555555')

plt.tight_layout(rect=(0,0.045,1,1))

ruta_png = ruta_salida / 'mapa_densidad_fatales_cdmx_2024.png'
ruta_svg = ruta_salida / 'mapa_densidad_fatales_cdmx_2024.svg'

fig.savefig(ruta_png,dpi=300,bbox_inches='tight',facecolor='white')
fig.savefig(ruta_svg,bbox_inches='tight',facecolor='white')

plt.show()

print(f'Mapa guardado en:\n- {ruta_png}\n- {ruta_svg}')

### Mapa interactivo: Incidentes fatales

Crearenos también un mapa interactivo para poder explorar las concentraciones a distintos niveles de zoom. Como en el mapa general, el efecto del calor depende del nivel de zoom y no equivale a una tasa de riesgo.


In [ ]:
# Incidentes con al menos una persona fallecida
df_mapa_fatales = df_mapa[df_mapa['personas_fallecidas'] > 0].copy()

mapa_fatales = folium.Map(
    location=centro,
    zoom_start=11,
    tiles=None,
    control_scale=True
)

folium.TileLayer(
    tiles=(
        "https://server.arcgisonline.com/ArcGIS/rest/services/"
        "World_Street_Map/MapServer/tile/{z}/{y}/{x}"
    ),
    attr="Tiles © Esri",
    name="Esri World Street Map"
).add_to(mapa_fatales)

HeatMap(
    data=df_mapa_fatales[['latitud','longitud']].values.tolist(),
    radius=11,
    blur=15,
    min_opacity=0.25,
    max_zoom=15,
    name='Incidentes fatales'
).add_to(mapa_fatales)

folium.LayerControl().add_to(mapa_fatales)

display(mapa_fatales)

In [ ]:
# Guardamos el mapa interactivo
ruta_html_fatales = ruta_mapas / 'fatal_incident_density_2024.html'
mapa_fatales.save(str(ruta_html_fatales))

print(f'Mapa guardado en:\n{ruta_html_fatales}')

### ¿Dónde hay una mayor proporción de incidentes fatales?

Crearemos un mapa que nos permita ver en qué zonas los incidentes fatales representan una proporción mayor de los incidentes registrados. Esto nos permitirá ver lugares que quizá no destacan por tener muchos accidentes, pero sí por presentar una mayor proporción de consecuencias fatales.

La idea matemática es sencilla:

$$
P(x,y)=
\frac{D_{\mathrm{fatal}}(x,y)}
{D_{\mathrm{total}}(x,y)}
\times100
$$

Si una zona registra únicamente un incidente y este fue fatal, podríamos obtener una proporción del 100%. Para evitar que esos casos dominen visualmente, mostraremos únicamente las zonas con un mínimo de 10 incidentes locales equivalentes, calculados a partir de la densidad suavizada. Recordemos que esto es un filtro de estabilidad para la visualización, no una afirmación de que haya exactamente diez incidentes dentro de un círculo.

In [ ]:
# Mismo suavizado espacial para ambas distribuciones
suavizado_m = 500
sigma = suavizado_m/resolucion_m

conteos_total,_,_ = np.histogram2d(
    gdf_puntos.geometry.y,gdf_puntos.geometry.x,
    bins=[bordes_y,bordes_x]
)

conteos_fatales,_,_ = np.histogram2d(
    gdf_fatales.geometry.y,gdf_fatales.geometry.x,
    bins=[bordes_y,bordes_x]
)

densidad_total = gaussian_filter(conteos_total,sigma=sigma)
densidad_fatales = gaussian_filter(conteos_fatales,sigma=sigma)

# Número equivalente de incidentes bajo el núcleo gaussiano
incidentes_locales = densidad_total*2*np.pi*sigma**2
min_incidentes = 10

# Proporción local, solo donde existe suficiente información
proporcion_fatal = np.divide(
    densidad_fatales,densidad_total,
    out=np.zeros_like(densidad_total),
    where=densidad_total>0
)*100

proporcion_visible = np.ma.masked_where(
    incidentes_locales<min_incidentes,
    proporcion_fatal
)

valores = proporcion_visible.compressed()
vmax = np.percentile(valores,95) if len(valores) else 1
vmax = max(vmax,1)

cmap = plt.colormaps['YlOrRd'].copy()
cmap.set_bad((0,0,0,0))

fig,ax = plt.subplots(figsize=(9,10),facecolor='white')
ax.set_xlim(bordes_x[0],bordes_x[-1])
ax.set_ylim(bordes_y[0],bordes_y[-1])

# Mapa base
proveedores = [
    ('Esri World Street Map',cx.providers.Esri.WorldStreetMap),
    ('OpenStreetMap',cx.providers.OpenStreetMap.Mapnik)
]

for nombre,proveedor in proveedores:
    try:
        cx.add_basemap(
            ax,source=proveedor,crs=crs_metrico,
            zoom=11,attribution_size=6
        )
        print('Mapa base:',nombre)
        break
    except Exception as error:
        print(f'No se pudo cargar {nombre}: {error}')
else:
    plt.close(fig)
    raise RuntimeError('No se pudo cargar ningún mapa base.')

# Proporción local de incidentes fatales
img = ax.imshow(
    proporcion_visible,
    extent=[bordes_x[0],bordes_x[-1],bordes_y[0],bordes_y[-1]],
    origin='lower',
    cmap=cmap,
    norm=Normalize(vmin=0,vmax=vmax),
    alpha=0.75,
    zorder=2
)

gdf_alcaldias.boundary.plot(
    ax=ax,color='#4D5866',
    linewidth=0.65,alpha=0.7,zorder=3
)

ax.set_title(
    'CIUDAD DE MÉXICO\nProporción local de incidentes fatales · 2024',
    fontsize=15,pad=15
)

ax.set_axis_off()

cbar = fig.colorbar(img,ax=ax,fraction=0.035,pad=0.02,extend='max')
cbar.set_label('Incidentes fatales respecto al total (%)',fontsize=9)
cbar.ax.tick_params(labelsize=8)

# Barra de escala
barra_x = bordes_x[0]+0.07*(bordes_x[-1]-bordes_x[0])
barra_y = bordes_y[0]+0.07*(bordes_y[-1]-bordes_y[0])

ax.plot(
    [barra_x,barra_x+10000],[barra_y,barra_y],
    color='#30343B',linewidth=3,zorder=5
)

ax.text(
    barra_x+5000,barra_y+700,'10 km',
    ha='center',fontsize=9,zorder=5,
    bbox={'facecolor':'white','edgecolor':'none','alpha':0.85,'pad':2}
)

fig.text(0.5,0.025, 'Fuente: SSC CDMX (2024) · Límites: catálogo de colonias',
         ha='center',fontsize=8,color='#555555')
fig.text(0.5,0.045, 'Densidad suavizada mediante filtro gaussiano (σ = 500 m)',
         ha='center',fontsize=8,color='#555555')

plt.tight_layout(rect=(0,0.045,1,1))

ruta_png = ruta_salida/'mapa_proporcion_fatal_cdmx_2024.png'
ruta_svg = ruta_salida/'mapa_proporcion_fatal_cdmx_2024.svg'

fig.savefig(ruta_png,dpi=300,bbox_inches='tight',facecolor='white')
fig.savefig(ruta_svg,bbox_inches='tight',facecolor='white')

plt.show()

print(f'Mapa guardado en:\n- {ruta_png}\n- {ruta_svg}')
print(f'Porcentaje máximo visible: {valores.max():.2f}%' if len(valores) else 'Sin zonas suficientes')

### Proporción local de incidentes fatales

A diferencia del mapa de frecuencia general, las zonas con mayor proporción de incidentes fatales aparecen distribuidas en el oriente, sur y en la periferia de la ciudad, mientras que la zona central presenta proporciones menores.

Esto nos muestra que **las zonas con mayores concentraciones de incidentes no necesariamente coinciden con las zonas donde hay una mayor proporción de fallecimientos**.

In [ ]:
from shapely.geometry import Polygon
from branca.colormap import LinearColormap

# Hexágonos de aproximadamente 1 km de ancho
radio_hex = 1000/np.sqrt(3)
ancho_hex = np.sqrt(3)*radio_hex
alto_fila = 1.5*radio_hex

xmin,ymin,xmax,ymax = gdf_puntos.total_bounds

hexagonos = []

for j,cy in enumerate(np.arange(ymin-alto_fila,ymax+alto_fila,alto_fila)):
    desplazamiento = ancho_hex/2 if j%2 else 0

    for centro_x in np.arange(xmin-ancho_hex,xmax+ancho_hex,ancho_hex):
        centro_x += desplazamiento
        angulos = np.deg2rad(np.arange(0,360,60)+30)

        vertices = [
            (centro_x+radio_hex*np.cos(a),cy+radio_hex*np.sin(a))
            for a in angulos
        ]

        hexagonos.append(Polygon(vertices))

gdf_hex = gpd.GeoDataFrame(
    geometry=hexagonos,
    crs=crs_metrico
)

# Asociamos cada incidente con un hexágono
puntos_hex = gdf_puntos[['folio','geometry']].merge(
    df_mapa[['folio','personas_fallecidas']],
    on='folio',
    validate='one_to_one'
)

puntos_hex['fatal'] = puntos_hex['personas_fallecidas'] > 0

union_hex = gpd.sjoin(
    puntos_hex,
    gdf_hex[['geometry']],
    how='inner',
    predicate='within'
)

resumen_hex = union_hex.groupby('index_right').agg(
    incidentes=('folio','count'),
    fatales=('fatal','sum')
)

gdf_hex = gdf_hex.join(resumen_hex)
gdf_hex[['incidentes','fatales']] = (
    gdf_hex[['incidentes','fatales']].fillna(0).astype(int)
)

gdf_hex = gdf_hex[gdf_hex['incidentes'] > 0].copy()

gdf_hex['pct_fatales'] = (
    gdf_hex['fatales']/gdf_hex['incidentes']*100
)

min_incidentes_hex = 15
gdf_hex['suficiente'] = gdf_hex['incidentes'] >= min_incidentes_hex

print('Hexágonos con incidentes:',len(gdf_hex))
print('Incidentes asignados:',gdf_hex['incidentes'].sum())
print('Incidentes fatales asignados:',gdf_hex['fatales'].sum())

# Preparación para Folium
gdf_hex['porcentaje'] = gdf_hex['pct_fatales'].map(lambda x:f'{x:.2f}%')
gdf_hex['clasificacion'] = np.where(
    gdf_hex['suficiente'],'Con datos suficientes','Menos de 15 incidentes'
)

gdf_hex_web = gdf_hex.to_crs('EPSG:4326')

escala_hex = LinearColormap(
    colors=['#FFF3C4','#F7B267','#E76F51','#8F2948'],
    vmin=0,
    vmax=10,
    caption='Incidentes fatales por hexágono (%)'
)

mapa_proporcion_hex = folium.Map(
    location=centro,
    zoom_start=11,
    tiles=None,
    control_scale=True
)

folium.TileLayer(
    tiles=(
        'https://server.arcgisonline.com/ArcGIS/rest/services/'
        'World_Street_Map/MapServer/tile/{z}/{y}/{x}'
    ),
    attr='Tiles © Esri',
    name='Esri World Street Map'
).add_to(mapa_proporcion_hex)

folium.GeoJson(
    gdf_hex_web,
    name='Proporción fatal por hexágono',
    style_function=lambda feature:{
        'fillColor': escala_hex(feature['properties']['pct_fatales'])
        if feature['properties']['suficiente'] else '#A9ADB4',
        'color':'#565D65',
        'weight':0.6,
        'fillOpacity':0.65
    },
    highlight_function=lambda feature:{
        'color':'#20242B',
        'weight':2,
        'fillOpacity':0.85
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['incidentes','fatales','porcentaje','clasificacion'],
        aliases=[
            'Incidentes:',
            'Incidentes fatales:',
            'Proporción fatal:',
            'Cobertura:'
        ],
        labels=True,
        sticky=True
    )
).add_to(mapa_proporcion_hex)

escala_hex.add_to(mapa_proporcion_hex)
folium.LayerControl().add_to(mapa_proporcion_hex)

display(mapa_proporcion_hex)

In [ ]:
# Guardamos el mapa por hexágonos
ruta_html_proporcion = ruta_mapas / 'fatal_incident_proportion_2024.html'
mapa_proporcion_hex.save(str(ruta_html_proporcion))

print(f'Mapa guardado en:\n{ruta_html_proporcion}')

### Proporción fatal por hexágonos

Para complementar, dividimos la ciudad en hexágonos de aproximadamente 1 km de ancho y calculamos la proporción de incidentes fatales dentro de cada uno. El mapa interactivo permite consultar tanto el número total de incidentes como aquellos que registraron fallecimientos.

Los hexágonos con menos de 15 incidentes aparecen en gris para distinguir las zonas con pocos registros. Este mapa nos permite explorar diferencias locales sin depender del suavizado gaussiano, aunque sus resultados también dependen del tamaño y la posición de la cuadrícula.

In [ ]:
# Identificamos los incidentes clasificados como motocicleta
folios_moto = df_vehiculos.loc[
    df_vehiculos['tipo_vehiculo_grupo']=='Motocicleta','folio'
]

gdf_motos = gdf_puntos[
    gdf_puntos['folio'].isin(folios_moto)
].copy()

print(f'Incidentes con motocicleta: {len(gdf_motos):,}')

# Densidad espacial con el mismo suavizado de 500 m
conteos_motos,_,_ = np.histogram2d(
    gdf_motos.geometry.y,gdf_motos.geometry.x,
    bins=[bordes_y,bordes_x]
)

densidad_motos = gaussian_filter(
    conteos_motos,sigma=sigma
) / (resolucion_m/1000)**2

umbral_motos = 0.5
valores_motos = densidad_motos[densidad_motos>=umbral_motos]
vmax_motos = np.percentile(valores_motos,99)

densidad_motos_visible = np.ma.masked_less(densidad_motos,umbral_motos)

cmap_motos = LinearSegmentedColormap.from_list(
    'motos_azul_violeta',
    ['#E0F2FE','#7DD3FC','#3B82F6','#6366F1','#4C1D95']
)

cmap_motos.set_bad((0,0,0,0))

fig,ax = plt.subplots(figsize=(9,10),facecolor='white')
ax.set_xlim(bordes_x[0],bordes_x[-1])
ax.set_ylim(bordes_y[0],bordes_y[-1])

# Mapa base
proveedores = [
    ('Esri World Street Map',cx.providers.Esri.WorldStreetMap),
    ('OpenStreetMap',cx.providers.OpenStreetMap.Mapnik)
]

for nombre,proveedor in proveedores:
    try:
        cx.add_basemap(
            ax,source=proveedor,crs=crs_metrico,
            zoom=11,attribution_size=6
        )
        print('Mapa base:',nombre)
        break
    except Exception as error:
        print(f'No se pudo cargar {nombre}: {error}')
else:
    plt.close(fig)
    raise RuntimeError('No se pudo cargar ningún mapa base.')

img = ax.imshow(
    densidad_motos_visible,
    extent=[bordes_x[0],bordes_x[-1],bordes_y[0],bordes_y[-1]],
    origin='lower',
    cmap=cmap_motos,
    norm=Normalize(vmin=umbral_motos,vmax=vmax_motos),
    alpha=0.7,zorder=2
)

gdf_alcaldias.boundary.plot(
    ax=ax,color='#4D5866',
    linewidth=0.65,alpha=0.7,zorder=3
)

ax.set_title(
    'CIUDAD DE MÉXICO\nDensidad de incidentes con motocicletas · 2024',
    fontsize=15,pad=15
)

ax.set_axis_off()

cbar = fig.colorbar(img,ax=ax,fraction=0.035,pad=0.02,extend='max')
cbar.set_label('Densidad espacial suavizada (incidentes/km²)',fontsize=9)
cbar.ax.tick_params(labelsize=8)

# Barra de escala
barra_x = bordes_x[0]+0.07*(bordes_x[-1]-bordes_x[0])
barra_y = bordes_y[0]+0.07*(bordes_y[-1]-bordes_y[0])

ax.plot(
    [barra_x,barra_x+10000],[barra_y,barra_y],
    color='#30343B',linewidth=3,zorder=5
)

ax.text(
    barra_x+5000,barra_y+700,'10 km',
    ha='center',fontsize=9,zorder=5,
    bbox={'facecolor':'white','edgecolor':'none','alpha':0.85,'pad':2}
)

fig.text(0.5,0.025, 'Fuente: SSC CDMX (2024) · Límites: catálogo de colonias',
         ha='center',fontsize=8,color='#555555')
fig.text(0.5,0.045, 'Densidad suavizada mediante filtro gaussiano (σ = 500 m)',
         ha='center',fontsize=8,color='#555555')

plt.tight_layout(rect=(0,0.065,1,1))

ruta_png = ruta_salida/'mapa_densidad_motocicletas_cdmx_2024.png'
ruta_svg = ruta_salida/'mapa_densidad_motocicletas_cdmx_2024.svg'

fig.savefig(ruta_png,dpi=300,bbox_inches='tight',facecolor='white')
fig.savefig(ruta_svg,bbox_inches='tight',facecolor='white')

plt.show()

print(f'Mapa guardado en:\n- {ruta_png}\n- {ruta_svg}')

### Distribución espacial de incidentes con motocicletas

Ya que las motocicletas estuvieron involucradas en 17,391 incidentes, que equivalen al 56.73% del total se revisó en que partes de la ciudad se concentraban incidentes con su participación. Su distribución espacial presenta una concentración importante en la zona central de la ciudad, con algunas zonas que se extienden hacia el oriente y sur.

El patrón se parece al observado en el mapa general donde se incluyeron todos los tipos de vehículos, algo razonable considerando la elevada participación de las motocicletas en nuestros registros.

In [ ]:
# Participación de motocicletas por hexágono
folios_moto_set = set(folios_moto)

motos_hex = union_hex.assign(
    moto=union_hex['folio'].isin(folios_moto_set)
)

resumen_motos_hex = motos_hex.groupby('index_right').agg(
    incidentes_moto=('moto','sum')
)

gdf_hex_motos = gdf_hex.join(resumen_motos_hex)
gdf_hex_motos['incidentes_moto'] = (
    gdf_hex_motos['incidentes_moto'].fillna(0).astype(int)
)

gdf_hex_motos['pct_motos'] = (
    gdf_hex_motos['incidentes_moto']
    / gdf_hex_motos['incidentes']*100
)

gdf_hex_motos['participacion'] = (
    gdf_hex_motos['pct_motos'].map(lambda x:f'{x:.1f}%')
)

gdf_hex_motos_web = gdf_hex_motos.to_crs('EPSG:4326')

# Escala de participación
escala_motos = LinearColormap(
    colors=['#E0F2FE','#7DD3FC','#3B82F6','#6366F1','#4C1D95'],
    vmin=0,vmax=100,
    caption='Participación de motocicletas (%)'
)

mapa_participacion_motos = folium.Map(
    location=centro,zoom_start=11,
    tiles=None,control_scale=True
)

folium.TileLayer(
    tiles=(
        'https://server.arcgisonline.com/ArcGIS/rest/services/'
        'World_Street_Map/MapServer/tile/{z}/{y}/{x}'
    ),
    attr='Tiles © Esri',
    name='Esri World Street Map'
).add_to(mapa_participacion_motos)

folium.GeoJson(
    gdf_hex_motos_web,
    name='Participación de motocicletas',
    style_function=lambda feature:{
        'fillColor':escala_motos(feature['properties']['pct_motos'])
        if feature['properties']['suficiente'] else '#A9ADB4',
        'color':'#565D65',
        'weight':0.6,
        'fillOpacity':0.7
    },
    highlight_function=lambda feature:{
        'color':'#20242B',
        'weight':2,
        'fillOpacity':0.9
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['incidentes','incidentes_moto','participacion','clasificacion'],
        aliases=[
            'Incidentes totales:',
            'Con motocicleta:',
            'Participación:',
            'Cobertura:'
        ],
        labels=True,sticky=True
    )
).add_to(mapa_participacion_motos)

escala_motos.add_to(mapa_participacion_motos)
folium.LayerControl().add_to(mapa_participacion_motos)

display(mapa_participacion_motos)

In [ ]:
# Guardamos el mapa de participación
ruta_html_motos = ruta_mapas/'motorcycle_share_hexagons_2024.html'
mapa_participacion_motos.save(str(ruta_html_motos))

print(f'Mapa guardado en:\n{ruta_html_motos}')
print(f'Tamaño: {ruta_html_motos.stat().st_size/1024/1024:.2f} MB')

### Participación de motocicletas por zona

Buscando una perspectiva distinta, calculamos la participación de las motocicletas dentro de hexágonos de aproximadamente 1 km de ancho. Los hexágonos con menos de 15 incidentes aparecen en gris; los demás permiten consultar conteos y porcentajes.

In [ ]:
df_alcaldias_motos = df_mapa[['folio','alcaldia_utilizada']].merge(
    df_vehiculos[['folio','tipo_vehiculo_grupo']],
    on='folio',validate='one_to_one'
)

df_alcaldias_motos['motocicleta'] = (
    df_alcaldias_motos['tipo_vehiculo_grupo']=='Motocicleta'
)

resumen_alcaldias_motos = (
    df_alcaldias_motos
    .groupby('alcaldia_utilizada')
    .agg(
        incidentes=('folio','count'),
        incidentes_moto=('motocicleta','sum')
    )
)

resumen_alcaldias_motos['pct_motos'] = (
    resumen_alcaldias_motos['incidentes_moto']
    /resumen_alcaldias_motos['incidentes']*100
)

display(resumen_alcaldias_motos.sort_values('pct_motos',ascending=False).round(2))

### Participación de motocicletas por alcaldía

Como ya lo habíamos preevisto las motocicletas representan **más de la mitad de los incidentes en cada una de las 16 alcaldías**. Tláhuac (63.73%), Álvaro Obregón (62.08%) y La Magdalena Contreras (61.81%) presentan las participaciones más altas, mientras que Benito Juárez (50.91%) y Miguel Hidalgo (53.37%), las más bajas.

El predominio del grupo es general, la tabla funciona como complemento del mapa de hexágonos permitiendo ver los conteos.


In [ ]:
resumen_alcaldias_fatales = (
    df_mapa
    .groupby('alcaldia_utilizada')
    .agg(
        incidentes=('folio','count'),
        incidentes_fatales=('personas_fallecidas',lambda x:(x>0).sum()),
        personas_fallecidas=('personas_fallecidas','sum')
    )
)

resumen_alcaldias_fatales['pct_fatales'] = (
    resumen_alcaldias_fatales['incidentes_fatales']
    /resumen_alcaldias_fatales['incidentes']*100
)

display(
    resumen_alcaldias_fatales
    .sort_values('pct_fatales',ascending=False)
    .round(2)
)

Al revisar el porcentaje de incidentes fatales podemos ver que: **Cuauhtémoc registra más incidentes en total, pero Iztapalapa tiene una proporción fatal mayor (2.95% frente a 0.81%)**. En números, hubo 118 incidentes fatales en Iztapalapa y 35 en Cuauhtémoc.

Crearemos visualizaciones de cada alcaldía que nos permitan observar dónde se concentran los incidentes con lesionados y localizar los eventos fatales sin mezclar ambas representaciones.


In [ ]:
# Mapas de alcaldias con incidentes y consecuencias
claves_alcaldias = {
    'AZCAPOTZALCO':2,'COYOACAN':3,'CUAJIMALPA DE MORELOS':4,
    'GUSTAVO A. MADERO':5,'IZTACALCO':6,'IZTAPALAPA':7,
    'LA MAGDALENA CONTRERAS':8,'MILPA ALTA':9,'ALVARO OBREGON':10,
    'TLAHUAC':11,'TLALPAN':12,'XOCHIMILCO':13,
    'BENITO JUAREZ':14,'CUAUHTEMOC':15,'MIGUEL HIDALGO':16,
    'VENUSTIANO CARRANZA':17
}

ruta_atlas = ruta_salida/'alcaldias'
ruta_atlas.mkdir(parents=True,exist_ok=True)

gdf_eventos = gdf_puntos.merge(
    df_mapa[['folio','alcaldia_utilizada','personas_lesionadas','personas_fallecidas']],
    on='folio',validate='one_to_one'
)

# Cuadrícula y colores
centros_x = (bordes_x[:-1]+bordes_x[1:])/2
centros_y = (bordes_y[:-1]+bordes_y[1:])/2
xx,yy = np.meshgrid(centros_x,centros_y)
puntos_grilla = gpd.GeoSeries(gpd.points_from_xy(xx.ravel(),yy.ravel()),crs=crs_metrico)
extension = [bordes_x[0],bordes_x[-1],bordes_y[0],bordes_y[-1]]

cmap_atlas = LinearSegmentedColormap.from_list(
    'densidad_lesionados',['#FFE082','#FFB74D','#FB8C00','#E65100']
)
cmap_atlas.set_bad((0,0,0,0))


def crear_mapa_alcaldia(alcaldia,clave):
    # Incidentes y geometría de la alcaldía
    gdf_local = gdf_eventos[gdf_eventos['alcaldia_utilizada']==alcaldia]
    gdf_lesionados = gdf_local[gdf_local['personas_lesionadas']>0]
    gdf_fatales = gdf_local[gdf_local['personas_fallecidas']>0]
    limite = gdf_alcaldias[gdf_alcaldias['CVEDT'].astype(int)==clave]
    if limite.empty:
        raise ValueError(f'No se encontró el límite de {alcaldia}')

    territorio = limite.geometry.union_all()
    xmin,ymin,xmax,ymax = territorio.bounds
    margen = 1500

    # Densidad de lesionados con suavizado de 500 m
    conteos,_,_ = np.histogram2d(
        gdf_lesionados.geometry.y,gdf_lesionados.geometry.x,bins=[bordes_y,bordes_x]
    )
    densidad = gaussian_filter(conteos,sigma=sigma)/(resolucion_m/1000)**2

    mascara_territorio = puntos_grilla.intersects(territorio).to_numpy().reshape(densidad.shape)
    valores_locales = densidad[mascara_territorio]
    umbral_local = np.percentile(valores_locales,75)
    vmax_local = np.percentile(valores_locales,99)
    densidad_visible = np.ma.masked_where(
        ~mascara_territorio | (densidad<umbral_local),densidad
    )

    # Mapa y densidad
    fig,ax = plt.subplots(figsize=(9,7),facecolor='white')
    ax.set_xlim(xmin-margen,xmax+margen)
    ax.set_ylim(ymin-margen,ymax+margen)
    cx.add_basemap(ax,source=cx.providers.Esri.WorldStreetMap,
                   crs=crs_metrico,zoom=13,attribution_size=6)

    img = ax.imshow(
        densidad_visible,extent=extension,origin='lower',cmap=cmap_atlas,
        norm=Normalize(vmin=umbral_local,vmax=vmax_local),alpha=0.65,zorder=2
    )

    # Contorno exterior sin huecos internos
    poligonos = list(territorio.geoms) if territorio.geom_type=='MultiPolygon' else [territorio]
    for poligono in poligonos:
        if poligono.area>=250000:
            ax.plot(*poligono.exterior.xy,color='#30343B',linewidth=1.4,zorder=3)

    # Puntos de incidentes fatales
    ax.scatter(
        gdf_fatales.geometry.x,gdf_fatales.geometry.y,
        s=30,color='#C56C54',edgecolor='white',linewidth=0.6,
        alpha=0.95,zorder=4,label='Incidente con fallecimiento'
    )

    cbar = fig.colorbar(img,ax=ax,fraction=0.035,pad=0.025,shrink=0.8,extend='max')
    cbar.set_label('Densidad espacial suavizada (incidentes/km²)',fontsize=9)
    cbar.ax.tick_params(labelsize=8)

    ax.set_title(f'{alcaldia}\nIncidentes con lesionados y ubicaciones fatales · 2024',
                 fontsize=14,pad=15)
    ax.set_axis_off()
    ax.legend(loc='upper right',fontsize=9)

    # Notas posicionadas respecto al borde inferior del mapa
    fig.tight_layout(rect=(0,0.08,1,1))
    fig.canvas.draw()
    pos = ax.get_position()
    centro_mapa = (pos.x0+pos.x1)/2

    fig.text(centro_mapa,pos.y0-0.025,
             'Filtro gaussiano (σ = 500 m) · Concentraciones superiores al percentil 75 local',
             ha='center',va='top',fontsize=8,color='#555555')
    fig.text(centro_mapa,pos.y0-0.048,
             'Fuente: SSC CDMX (2024) · Límites: catálogo de colonias',
             ha='center',va='top',fontsize=8,color='#555555')

    # Guardamos PNG y SVG
    nombre_archivo = alcaldia.lower().replace(' ','_').replace('.','')
    ruta_png = ruta_atlas/f'{nombre_archivo}_incidentes_2024.png'
    ruta_svg = ruta_atlas/f'{nombre_archivo}_incidentes_2024.svg'
    fig.savefig(ruta_png,dpi=300,bbox_inches='tight',facecolor='white')
    fig.savefig(ruta_svg,bbox_inches='tight',facecolor='white')
    return fig,len(gdf_local),len(gdf_fatales)


# Exportamos los 16 mapas y vemos Izt
fig_muestra = None
resultados_atlas = []

for alcaldia,clave in claves_alcaldias.items():
    fig,n_incidentes,n_fatales = crear_mapa_alcaldia(alcaldia,clave)
    resultados_atlas.append({
        'alcaldia':alcaldia,'incidentes':n_incidentes,'incidentes_fatales':n_fatales
    })
    if alcaldia=='IZTAPALAPA':
        fig_muestra = fig
    else:
        plt.close(fig)

# Verificamos la cobertura del atlas
assert sum(r['incidentes'] for r in resultados_atlas)==len(df_mapa)
assert sum(r['incidentes_fatales'] for r in resultados_atlas)==(df_mapa['personas_fallecidas']>0).sum()

plt.figure(fig_muestra.number)
plt.show()
print(f'Atlas generado: {len(resultados_atlas)} alcaldías')
print(f'Archivos exportados: {len(resultados_atlas)*2} (PNG y SVG)')
print(f'Los demás mapas fueron guardados en:\n{ruta_atlas}')


### Atlas de incidentes viales por alcaldía

Estos **16 mapas** nos permiten ver la densidad de los incidentes con lesionados y ubicar los que tuvieron consecuencias fatales. Las manchas amarillas y naranjas indican las concentraciones de incidentes que tuvieron lesionados y los puntos rojos señalan los lugares donde se registró al menos un fallecimiento.

Mostramos únicamente las concentraciones por encima del percentil 75 local para evitar saturar demasiado el mapa y permitir distinguir las calles. **El mapa de color se ajusta por alcaldía**, nos ayuda a comparar zonas dentro de cada alcaldía, pero no intensidades absolutas entre todas ya que se calcula con base a los totales de cada alcaldía. Los puntos, en cambio, si representan ubicaciones registradas.

Los límites territoriales son una aproximación elaborada a partir del catálogo de colonias. Los **16 mapas**, se exportarán en PNG y SVG, pueden consultarse en `content/maps/static/alcaldias/`, aquí mostramos únicamente Iztapalapa para no saturarnos de figuras.

## ¿Qué nos llevamos de esta exploración?

Comenzamos revisando generalidades y conforme avanzamos descubrimos que al cambiar el denominador con el que comparamos o la escala que utilizamos, también cambia bastante lo que podemos observar.

A pesar de las limitaciones en los registros, como el que solamente aparezca un vehículo por registro o el que no haya información completa de todas las personas involucradas en los incidentes. Pudimos realizar bastantes hallazgos como:
- Las **motocicletas aparecen en el 56.73% de los folios**, y su presencia supera el 50% en todas las alcaldías.

- El número de incidentes no nos cuenta por sí solo cuántas personas resultan afectadas. Como el caso de los registros asociados con **transporte colectivo**, que presentaron con mayor frecuencia **eventos con múltiples personas lesionadas**.

- La **madrugada destaca por presentar una mayor proporción de incidentes fatales entre los incidentes registrados**. Este resultado sugiere que las consecuencias de los incidentes ocurridos en ese horario son proporcionalmente más graves, aunque no podemos afirmar que circular de madrugada implique un mayor riesgo de sufrir un accidente si no conocemos el volumen de vehículos que circulan en ese horario.

- Las diferencias también aparecen en la infraestructura vial y el territorio. Las vías secundarias acumulan muchos incidentes, mientras que los accesos carreteros tienen una mayor proporción fatal. Y aunque la alcaldía **Cuauhtémoc registra más incidentes que Iztapalapa**, esta última presenta más incidentes fatales y una proporción fatal superior.

- Finalmente, los mapas nos ayudaron a rmostrar por qué **concentración de incidentes y gravedad no son lo mismo**.

### Lo que todavía no podemos afirmar

Dado que estos datos corresponden únicamente a los incidentes registrados en 2024 y carecemos de registros de otros años o del total de desplazamientos en la ciudad, no es posible establecer un nexo causal que indique si cierto grupo de vehículos es más peligroso que otro, confirmar patrones de estacionalidad mensual o señalar relaciones deterministas.

Sin embargo, este análisis exploratorio aporta hallazgos clave que probablemente ayuden para orientar futuras investigaciones. En este sentido, al no contar con métricas de exposición (como volumen de viajes, aforo vehicular o kilómetros recorridos), los conteos y proporciones no pueden traducirse directamente en tasas de riesgo por zona. Por ende, tampoco es viable atribuir efectos causales a una vialidad, a la semaforización o a los periodos vacacionales basándonos únicamente en las asociaciones observadas.

Finalmente, en el plano espacial, las superficies suavizadas dependen de la resolución, del ancho de banda ($\sigma$) y de los umbrales visuales elegidos; estas densidades facilitan la identificación de zonas de concentración, pero no sustituyen las ubicaciones puntuales reales. Del mismo modo, las agregaciones por hexágonos o alcaldías están condicionadas por su propia unidad de corte (efecto de área modificable), por lo que sirven para visibilizar patrones espaciales, mas no para inferir relaciones causales.

## Siguiente etapa: Machine Learning

Ahora finalmente tenemos un punto de partida para lo siguiente, que será el **tercer notebook**. En él comenzaremos por definir una pregunta que podamos responder con las variables disponibles.

Una posibilidad es desarrollar un problema de clases desbalanceadas y buscar que un algoritmo pueda clasificar correctamente **si un incidente registrado  tuvo al menos un fallecimiento**. La exactitud por sí sola sería engañosa: compararemos modelos con una línea base y métricas como *precision*, *recall* y el área bajo la curva precisión–recobrado (*PR-AUC*).

En algoritmos de ML hay que tener que cuidado con la **fuga de información**. Características como `personas_fallecidas` definen la etiqueta, y otras como `prioridad` podrían reflejar consecuencias conocidas después del hecho y no deberían utilizarse como predictores. Revisaremos con cuidado qué variables estarían disponibles en el momento en que pretendemos hacer una predicción, y cómo separar los datos de entrenamiento y evaluación sin producir resultados artificialmente optimistas.

Después podremos explorar otros enfoques, incluida la identificación de perfiles o patrones espaciales. Por ahora cerramos este EDA con **preguntas concretas, variables que conocemos mejor y limitaciones que no debebemos olvidar al entrenar**.

## Anexo · Exportación de figuras

Para reutilizar nuestras gráficas, guardaremos sus imágenes en `content/figures/`, separadas de los mapas que ya se exportaron a `content/maps/`.

El anexo recupera las **gráficas que ya están guardadas en las salidas de este notebook** por lo que si se ejecuta sin estar guardado en el lugar adecuado puede no funcionar.


In [ ]:
# Guardamos las gráficas existentes
import base64
import json

ruta_figuras = ruta_proyecto/'content/figures'
ruta_figuras.mkdir(parents=True,exist_ok=True)

# Buscamos el notebook final dentro del proyecto
ruta_notebook = ruta_proyecto/'notebooks/02_Exploratory_Data_Analysis.ipynb'
if not ruta_notebook.is_file():
    candidatos = sorted({
        *ruta_proyecto.glob('02_Exploratory_Data_Analysis*.ipynb'),
        *(ruta_proyecto/'notebooks').glob('02_Exploratory_Data_Analysis*.ipynb')
    })
    if len(candidatos)!=1:
        raise FileNotFoundError(
            'No se identificó un único notebook 02. Guárdalo como '
            'notebooks/02_Exploratory_Data_Analysis.ipynb antes de exportar.'
        )
    ruta_notebook = candidatos[0]

with ruta_notebook.open(encoding='utf-8') as archivo:
    cuaderno = json.load(archivo)

# Identificamos las gráficas anteriores a la exploración geográfica
imagenes = []
for celda in cuaderno['cells']:
    fuente = celda.get('source',[])
    fuente = ''.join(fuente) if isinstance(fuente,list) else fuente
    if celda['cell_type']=='markdown' and '## Exploración geográfica' in fuente:
        break
    if celda['cell_type']!='code':
        continue
    for salida in celda.get('outputs',[]):
        imagen = salida.get('data',{}).get('image/png')
        if imagen:
            imagenes.append(''.join(imagen) if isinstance(imagen,list) else imagen)

# Nombres descriptivos en el mismo orden que las gráficas del EDA
nombres_figuras = [
    'incidentes_por_mes',
    'incidentes_por_alcaldia_y_mes',
    'distribucion_mensual_por_alcaldia',
    'incidentes_por_tipo_vehiculo',
    'porcentaje_por_tipo_vehiculo',
    'incidentes_por_grupo_vehiculo',
    'porcentaje_por_grupo_vehiculo',
    'variacion_mensual_grupos',
    'variacion_mensual_grupos_paneles',
    'lesionados_por_100_incidentes',
    'fallecidos_por_100_incidentes',
    'incidentes_cinco_o_mas_lesionados',
    'sexo_y_tipo_participante_lesionados',
    'composicion_lesionados_por_sexo',
    'tipo_participante_fallecidos',
    'sexo_incidentes_fatales',
    'distribucion_edades',
    'edades_por_participante_boxplot',
    'edades_por_participante_violin',
    'edades_por_participante_y_sexo',
    'menores_cinco_anios_lesionados',
    'incidentes_por_hora',
    'incidentes_por_dia_y_hora',
    'distribucion_horaria_relativa',
    'consecuencias_por_hora',
    'incidentes_fatales_dia_y_hora',
    'proporcion_fatal_dia_y_hora',
    'lesionados_por_bloque_horario',
    'fatales_por_bloque_horario',
    'evolucion_diaria_lesionados',
    'evolucion_diaria_fatales',
    'periodos_vacacionales_y_lesionados',
    'consecuencias_segun_vialidad',
    'consecuencias_segun_interseccion',
    'geometria_semaforo_y_fatales',
    'vehiculos_segun_vialidad',
    'composicion_vehicular_por_vialidad'
]

if len(imagenes)!=len(nombres_figuras):
    raise ValueError(
        f'Se encontraron {len(imagenes)} gráficas y se esperaban '
        f'{len(nombres_figuras)}. Revisa que guardaste todas las salidas '
        'del notebook final antes de exportar.'
    )

for numero,(imagen,nombre) in enumerate(zip(imagenes,nombres_figuras),start=1):
    ruta_imagen = ruta_figuras/f'{numero:02d}_{nombre}.png'
    ruta_imagen.write_bytes(base64.b64decode(imagen))

print(f'Gráficas exportadas: {len(imagenes)}')
print(f'Carpeta: {ruta_figuras}')
print('Los mapas estáticos e interactivos permanecen en content/maps/.')
